# Прогноз банкротства компаний — скоринговая карта vs градиентный бустинг (Google Colab)
**Практическое задание «ML-моделирование в финансах», вариант 5.** Данные: Kaggle «Company Bankruptcy Prediction» (Тайвань), файл `data.csv`.

**Как запускать:** выполняйте ячейки сверху вниз (*Среда выполнения → Выполнить всё*). Единственное, что нужно сделать руками, — в ячейке 0.2 выбрать файл `data.csv`.
Весь код проекта (модули `src/`, `config.yaml`) записывается ячейками ниже в рабочую папку Colab; в конце ячейка скачивает отчёт (`report.docx`), `model.pkl` и архив проекта для Git.

Структура: **0. Установка и код** → **1. Данные и разбиение** → **2. Признаки и базовая модель** → **3. Бустинг** → **4. Финансы, SHAP, PSI** → **5. Отчёт и скачивание**.

## 0. Установка и код проекта

In [ ]:
# 0.1 Зависимости. В Colab уже есть numpy/pandas/scikit-learn/lightgbm/matplotlib — доустанавливаем только недостающее.
!pip install -q lightgbm shap python-docx
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.makedirs("/content/bankruptcy-ml/src", exist_ok=True)
os.makedirs("/content/bankruptcy-ml/data", exist_ok=True)
os.chdir("/content/bankruptcy-ml")
print("рабочая папка:", os.getcwd())

In [ ]:
# 0.2 Данные: выберите файл data.csv (Kaggle «Company Bankruptcy Prediction»).
# Если файл лежит на Google Drive: from google.colab import drive; drive.mount("/content/drive")
# и скопируйте его в /content/bankruptcy-ml/data/data.csv
import os, shutil
if not os.path.exists("data/data.csv"):
    from google.colab import files
    up = files.upload()
    shutil.move(list(up)[0], "data/data.csv")
import pandas as pd
print(pd.read_csv("data/data.csv").shape, "(ожидается (6819, 96))")

In [ ]:
%%writefile config.yaml
seed: 42
data:
  path: data/data.csv
  target: "Bankrupt?"
  # В датасете нет даты отчёта: порядок строк используется как прокси времени (см. README, раздел «Ограничения»)
  time_proxy: row_order
split:
  train: 0.70
  valid: 0.15        # test = оставшиеся 15%
features:
  corr_duplicate_threshold: 0.98   # пары с |rho| >= порога считаются дублями (оставляем более сильный)
  clip_quantiles: [0.01, 0.99]     # границы винзоризации для новых признаков (считаются на train)
baseline:
  max_bins: 10
  min_bin_share: 0.05
  min_bin_bads: 5
  iv_min: 0.05
  corr_max: 0.70
  max_features: 15
  c_grid: [0.01, 0.03, 0.1, 0.3, 1.0]
  scorecard: {base_score: 600, base_odds: 50, pdo: 20}
boosting:
  engine: lightgbm                 # lightgbm | hist_gb (запасной движок, если lightgbm не установлен)
  early_stopping_rounds: 100
  max_estimators: 3000
  search: {n_iter: 30, n_splits: 5}
  # final_params перезаписывается автоматически после этапа 3 (python run_all.py)
  final_params:
    n_estimators: 2000
    learning_rate: 0.03
    num_leaves: 31
    min_child_samples: 100
    subsample: 0.8
    subsample_freq: 1
    colsample_bytree: 0.8
    reg_lambda: 1.0
    reg_alpha: 0.0
    scale_pos_weight: 1.0
calibration:
  method: isotonic
  pd_floor: 0.0005          # нижняя граница PD после калибровки
finance:
  lgd: 0.45                  # по условию задания
  ead: 1.0                   # экспозиция на компанию (нормировано)
  margin: 0.08               # ДОПУЩЕНИЕ: чистая маржа с одобренной «хорошей» компании, доля от EAD
  margin_sensitivity: [0.03, 0.05, 0.08, 0.12]
  check_shares: [0.05, 0.10, 0.20]   # доля компаний, отбираемых на проверку (для полноты)
  main_check_share: 0.10
  ead_money_example: 1000000          # условная EAD в тенге для иллюстрации «в деньгах»
stability:
  bins: 10
  psi_warn: 0.10
  psi_threshold: 0.25
  top_features: 5
bootstrap: {n: 1000}

In [ ]:
%%writefile src/__init__.py
# пакет src

In [ ]:
%%writefile src/data.py
"""Загрузка данных, разбиение по времени, аудит признаков (константы, дубли, подозрение на утечку)."""
from __future__ import annotations

import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score

ROOT = Path(__file__).resolve().parents[1]


def load_config(path: str | Path = ROOT / "config.yaml") -> dict:
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_data(cfg: dict) -> pd.DataFrame:
    path = ROOT / cfg["data"]["path"]
    if not path.exists():
        raise FileNotFoundError(
            f"Нет файла {path}. Скачайте data.csv с Kaggle «Company Bankruptcy Prediction» "
            "(fedesoriano) и положите в data/.")
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    # Порядок строк = прокси времени. Явный индекс «периода» делаем только для графиков/описания.
    df.insert(0, "obs_order", np.arange(len(df)))
    return df


def time_split(df: pd.DataFrame, train: float = 0.70, valid: float = 0.15):
    """train — первые 70%, valid — следующие 15%, test (out-of-time) — последние 15%."""
    df = df.sort_values("obs_order", kind="stable")
    n = len(df)
    a, b = int(n * train), int(n * (train + valid))
    return df.iloc[:a].copy(), df.iloc[a:b].copy(), df.iloc[b:].copy()


# ----- описание признаков -----
def describe_features(train: pd.DataFrame, target: str) -> pd.DataFrame:
    X = train.drop(columns=[target, "obs_order"])
    rows = []
    for c in X.columns:
        s = X[c]
        rows.append({"feature": c, "dtype": str(s.dtype), "missing_share": float(s.isna().mean()),
                     "n_unique": int(s.nunique()), "min": float(s.min()), "median": float(s.median()),
                     "max": float(s.max()),
                     "gini_univariate": abs(2 * roc_auc_score(train[target], s.fillna(s.median())) - 1)})
    return pd.DataFrame(rows)


def audit_features(train: pd.DataFrame, target: str, dup_thr: float = 0.98, leak_gini: float = 0.90):
    """Аудит ТОЛЬКО по train. Возвращает (список удаляемых признаков, таблица с причинами).

    Классы проблем:
      constant   — признак не несёт информации (<=1 уникального значения);
      duplicate  — |rho| >= dup_thr с другим признаком (оставляем более сильный по одномерному Gini на train);
      leak_suspect — одномерный Gini > leak_gini (подозрительно «слишком хорошо», кандидат на утечку).
    Признаки «после события» (сумма просрочки и т.п.) в этом датасете отсутствуют: все 95 показателей —
    коэффициенты годовой отчётности за период ДО банкротства (проверено по названиям, см. отчёт).
    """
    X = train.drop(columns=[target, "obs_order"])
    y = train[target]
    ug = {c: abs(2 * roc_auc_score(y, X[c]) - 1) for c in X.columns}
    removed = []
    for c in X.columns:
        if X[c].nunique() <= 1:
            removed.append({"feature": c, "reason": "constant", "detail": "единственное значение на train", "partner": ""})
    gone = {r["feature"] for r in removed}
    # однозначные «двойники» (|rho| ~ 1) по Спирмену: монотонные преобразования не ловим пирсоном
    corr = X.drop(columns=list(gone)).corr(method="spearman").abs()
    cols = list(corr.columns)
    for i, j in itertools.combinations(cols, 2):
        if i in gone or j in gone:
            continue
        if corr.loc[i, j] >= dup_thr:
            drop, keep = (i, j) if ug[i] < ug[j] else (j, i)
            gone.add(drop)
            removed.append({"feature": drop, "reason": "duplicate",
                            "detail": f"|rho_Spearman|={corr.loc[i, j]:.3f}, Gini {ug[drop]:.3f} < {ug[keep]:.3f}",
                            "partner": keep})
    for c, g in ug.items():
        if g > leak_gini and c not in gone:
            removed.append({"feature": c, "reason": "leak_suspect", "detail": f"одномерный Gini={g:.3f}", "partner": ""})
            gone.add(c)
    return sorted(gone), pd.DataFrame(removed)

In [ ]:
%%writefile src/features.py
"""Генерация признаков с экономическим смыслом. Все статистики (квантили) считаются ТОЛЬКО на train."""
from __future__ import annotations

import numpy as np
import pandas as pd

EPS = 0.05


def _u(s: pd.Series) -> pd.Series:
    """Коэффициент в [0, 1]: в датасете часть значений закодирована как 1e9–1e10 (выбросы/«заглушки»)."""
    return s.clip(0, 1)


# (имя, функция, экономическое обоснование)
FEATURE_SPECS = [
    ("cash_cover_gap",
     lambda d: _u(d["Cash/Total Assets"]) - _u(d["Current Liability to Assets"]),
     "Денежные средства минус краткосрочные обязательства (в долях активов): способность погасить "
     "ближайшие долги без продажи активов. Отрицательное значение — кассовый разрыв."),
    ("quick_cover_gap",
     lambda d: _u(d["Quick Assets/Total Assets"]) - _u(d["Current Liability to Assets"]),
     "Быстрые активы минус краткосрочные обязательства: расширенная ликвидность "
     "(аналог quick ratio в разностной форме, устойчив к делению на малые числа)."),
    ("roa_per_leverage",
     lambda d: _u(d["Net Income to Total Assets"]) / (_u(d["Debt ratio %"]) + EPS),
     "Рентабельность активов на единицу долговой нагрузки: прибыльность компании с поправкой на то, "
     "каким долгом она достигнута."),
    ("cfo_per_leverage",
     lambda d: _u(d["CFO to Assets"]) / (_u(d["Debt ratio %"]) + EPS),
     "Операционный денежный поток на единицу долга: банкротство определяется нехваткой денег, "
     "а не бухгалтерской прибылью."),
    ("interest_drag",
     lambda d: _u(d["Operating Profit Rate"]) - _u(d["Pre-tax net Interest Rate"]),
     "Разрыв между операционной рентабельностью и рентабельностью после процентов: "
     "доля маржи, которую съедает обслуживание долга."),
    ("tax_drag",
     lambda d: _u(d["After-tax net Interest Rate"]) - _u(d["Pre-tax net Interest Rate"]),
     "Налоговая нагрузка на прибыль после процентов (разрыв до/после налога): "
     "аномалии могут говорить о качестве отчётности."),
    ("retained_self_financing",
     lambda d: _u(d["Retained Earnings to Total Assets"]) * _u(d["Net worth/Assets"]),
     "Накопленная нераспределённая прибыль × доля собственного капитала: "
     "способность компании финансировать себя изнутри (буфер на убытки)."),
    ("short_debt_leverage",
     lambda d: _u(d["Current Liability to Liability"]) * _u(d["Debt ratio %"]),
     "Доля краткосрочных обязательств в долге × общая долговая нагрузка: риск рефинансирования — "
     "короткий долг в большом объёме нужно постоянно перекредитовывать."),
    ("fixed_asset_debt_burden",
     lambda d: _u(d["Fixed Assets to Assets"]) * _u(d["Debt ratio %"]),
     "Доля основных средств × долг: неликвидные активы, профинансированные долгом, "
     "трудно быстро превратить в деньги при стрессе."),
    ("leverage_effect",
     lambda d: _u(d["Net Income to Stockholder's Equity"]) - _u(d["Net Income to Total Assets"]),
     "ROE − ROA: эффект финансового рычага (насколько доходность капитала держится на заёмных деньгах)."),
    ("eps_to_book",
     lambda d: _u(d["Persistent EPS in the Last Four Seasons"]) / (_u(d["Net Value Per Share (B)"]) + 0.01),
     "Устойчивая прибыль на акцию к балансовой стоимости акции: доходность на собственный капитал, "
     "очищенная от сезонных всплесков (4 квартала)."),
    ("cashflow_vs_borrowing",
     lambda d: _u(d["Cash Flow to Liability"]) - _u(d["Borrowing dependency"]),
     "Покрытие обязательств денежным потоком минус зависимость от заёмных средств: "
     "хватает ли потока на обслуживание той доли долга, на которой держится бизнес."),
    ("margin_x_turnover",
     lambda d: _u(d["Gross Profit to Sales"]) * _u(d["Total Asset Turnover"]),
     "Валовая маржа × оборачиваемость активов (компонента модели Дюпона): "
     "бизнес-модель «высокая маржа или быстрый оборот» защищает от убытков."),
    ("working_capital_vs_debt",
     lambda d: _u(d["Working Capital to Total Assets"]) - _u(d["Debt ratio %"]),
     "Чистый оборотный капитал минус общий долг (в долях активов): запас «прочности» после обслуживания долга."),
]
NEW_FEATURES = [n for n, _, _ in FEATURE_SPECS] + ["stress_flag"]
STRESS_RATIONALE = ("Флаг двойного стресса: рентабельность активов в нижнем квартиле И долговая нагрузка в верхнем "
                    "квартиле (пороги по train). Одновременно слабая прибыль и высокий долг — классический "
                    "предвестник дефолта, нелинейная комбинация, которую линейная модель сама не соберёт.")


def feature_rationales() -> dict:
    d = {n: r for n, _, r in FEATURE_SPECS}
    d["stress_flag"] = STRESS_RATIONALE
    return d


class FeatureBuilder:
    """fit() — только на train; transform() — на любой выборке. Никаких статистик от validation/test."""

    def __init__(self, drop_cols=(), clip_q=(0.01, 0.99), target="Bankrupt?"):
        self.drop_cols = list(drop_cols)
        self.clip_q = clip_q
        self.target = target

    def _new(self, df: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame({n: f(df) for n, f, _ in FEATURE_SPECS}, index=df.index)
        out["stress_flag"] = ((_u(df["Net Income to Total Assets"]) <= self.roa_q25_)
                              & (_u(df["Debt ratio %"]) >= self.debt_q75_)).astype(float)
        return out

    def fit(self, train: pd.DataFrame):
        self.roa_q25_ = _u(train["Net Income to Total Assets"]).quantile(0.25)
        self.debt_q75_ = _u(train["Debt ratio %"]).quantile(0.75)
        new = self._new(train)
        self.bounds_ = {c: (new[c].quantile(self.clip_q[0]), new[c].quantile(self.clip_q[1]))
                        for c in new.columns if c != "stress_flag"}
        raw = [c for c in train.columns if c not in (self.target, "obs_order") and c not in self.drop_cols]
        self.raw_cols_ = raw
        self.feature_names_ = raw + NEW_FEATURES
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        new = self._new(df)
        for c, (lo, hi) in self.bounds_.items():
            new[c] = new[c].clip(lo, hi)
        X = pd.concat([df[self.raw_cols_], new], axis=1)
        return X[self.feature_names_].astype(float)

    def fit_transform(self, train):
        return self.fit(train).transform(train)

In [ ]:
%%writefile src/woe.py
"""Классическая скоринговая карта: WoE-биннинг + логистическая регрессия + шкала баллов."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


def _bin_stats(y, bins, k):
    n_bad = np.bincount(bins, weights=y, minlength=k)
    n_tot = np.bincount(bins, minlength=k).astype(float)
    return n_tot, n_bad


class WoEBinner:
    """Квантильный биннинг с объединением соседних корзин, пока в каждой не меньше min_share наблюдений
    и min_bads банкротов. WoE = ln(доля хороших / доля плохих), со сглаживанием +0.5."""

    def __init__(self, max_bins=10, min_share=0.05, min_bads=5):
        self.max_bins, self.min_share, self.min_bads = max_bins, min_share, min_bads

    def fit(self, X: pd.DataFrame, y):
        y = np.asarray(y)
        self.edges_, self.woe_, self.iv_, self.table_ = {}, {}, {}, {}
        G, B = (y == 0).sum(), (y == 1).sum()
        for c in X.columns:
            x = X[c].to_numpy(float)
            q = np.unique(np.quantile(x, np.linspace(0, 1, self.max_bins + 1)[1:-1]))
            while True:
                b = np.searchsorted(q, x, side="right")
                k = len(q) + 1
                tot, bad = _bin_stats(y, b, k)
                weak = (tot < self.min_share * len(y)) | (bad < self.min_bads)
                if not weak.any() or k <= 2:
                    break
                i = int(np.argmin(np.where(weak, tot, np.inf)))     # самая слабая корзина
                # сливаем с соседом, у которого меньше наблюдений
                nb = i - 1 if i == k - 1 else (i + 1 if i == 0 else (i - 1 if tot[i - 1] <= tot[i + 1] else i + 1))
                cut = min(i, nb)                                     # убираем границу между i и nb
                q = np.delete(q, cut)
            b = np.searchsorted(q, x, side="right")
            k = len(q) + 1
            tot, bad = _bin_stats(y, b, k)
            good = tot - bad
            woe = np.log(((good + 0.5) / (G + 0.5 * k)) / ((bad + 0.5) / (B + 0.5 * k)))
            iv = float(np.sum(((good + 0.5) / (G + 0.5 * k) - (bad + 0.5) / (B + 0.5 * k)) * woe))
            self.edges_[c], self.woe_[c], self.iv_[c] = q, woe, iv
            lo = np.concatenate([[-np.inf], q]); hi = np.concatenate([q, [np.inf]])
            self.table_[c] = pd.DataFrame({"feature": c, "bin": range(k), "lo": lo, "hi": hi, "n": tot.astype(int),
                                           "bads": bad.astype(int), "bad_rate": np.divide(bad, tot, out=np.zeros_like(bad), where=tot > 0), "woe": woe})
        return self

    def transform(self, X: pd.DataFrame, cols=None) -> pd.DataFrame:
        cols = cols or list(self.edges_)
        out = {c: self.woe_[c][np.searchsorted(self.edges_[c], X[c].to_numpy(float), side="right")] for c in cols}
        return pd.DataFrame(out, index=X.index)


class Scorecard:
    """WoE → отбор по IV и корреляции → логистическая регрессия → баллы."""

    def __init__(self, max_bins=10, min_share=0.05, min_bads=5, iv_min=0.05, corr_max=0.7, max_features=15, C=0.1,
                 base_score=600, base_odds=50, pdo=20, seed=42):
        self.p = dict(max_bins=max_bins, min_share=min_share, min_bads=min_bads)
        self.iv_min, self.corr_max, self.max_features, self.C = iv_min, corr_max, max_features, C
        self.base_score, self.base_odds, self.pdo, self.seed = base_score, base_odds, pdo, seed

    def fit(self, X: pd.DataFrame, y):
        self.binner_ = WoEBinner(**self.p).fit(X, y)
        iv = pd.Series(self.binner_.iv_).sort_values(ascending=False)
        cand = [c for c in iv.index if iv[c] >= self.iv_min]
        W = self.binner_.transform(X, cand)
        chosen = []
        for c in cand:                                    # жадно: сильные признаки первыми, дубли по корреляции отбрасываем
            if len(chosen) >= self.max_features:
                break
            if all(abs(np.corrcoef(W[c], W[s])[0, 1]) < self.corr_max for s in chosen):
                chosen.append(c)
        self.iv_series_ = iv
        y = np.asarray(y)
        while True:                                       # WoE = ln(good/bad): коэффициент обязан быть < 0 (знак логичен)
            self.lr_ = LogisticRegression(C=self.C, max_iter=2000, random_state=self.seed).fit(W[chosen], y)
            bad_sign = [c for c, b in zip(chosen, self.lr_.coef_[0]) if b > 0]
            if not bad_sign:
                break
            chosen.remove(bad_sign[-1])                   # убираем самый слабый по IV среди «неправильных»
        self.features_ = chosen
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        return self.lr_.predict_proba(self.binner_.transform(X, self.features_))[:, 1]

    def get_params(self, deep=False):
        return {}

    def points_table(self) -> pd.DataFrame:
        """Баллы за каждую корзину: score = offset − factor·logit(PD)."""
        factor = self.pdo / np.log(2)
        offset = self.base_score - factor * np.log(self.base_odds)
        n = len(self.features_)
        rows = []
        for j, c in enumerate(self.features_):
            t = self.binner_.table_[c].copy()
            coef = self.lr_.coef_[0][j]
            t["coef"] = coef
            t["points"] = -(coef * t["woe"] + self.lr_.intercept_[0] / n) * factor + offset / n
            rows.append(t)
        return pd.concat(rows, ignore_index=True)

    def score(self, X: pd.DataFrame) -> np.ndarray:
        p = np.clip(self.predict_proba(X), 1e-9, 1 - 1e-9)
        factor = self.pdo / np.log(2)
        return self.base_score - factor * np.log(self.base_odds) - factor * np.log(p / (1 - p))

In [ ]:
%%writefile src/train.py
"""Обучение: бустинг (LightGBM; запасной движок sklearn HistGradientBoosting), подбор параметров по времени,
ранняя остановка, калибровка. Никакого обычного KFold — только TimeSeriesSplit / validation."""
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit

try:                                    # основной движок — как требует задание
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:                     # только для проверки пайплайна в среде без lightgbm
    lgb, HAS_LGB = None, False

try:
    from sklearn.frozen import FrozenEstimator
except ImportError:                     # sklearn < 1.6
    FrozenEstimator = None


def resolve_engine(requested: str) -> str:
    if requested == "lightgbm" and not HAS_LGB:
        warnings.warn("lightgbm не установлен — используется запасной движок hist_gb (результаты НЕ финальные). "
                      "Установите зависимости: pip install -r requirements.txt", RuntimeWarning)
        return "hist_gb"
    return requested


SEARCH_SPACE = {
    "num_leaves": [4, 8, 16, 31],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "min_child_samples": [20, 50, 100],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.4, 0.6, 0.8, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0, 20.0],
    "scale_pos_weight": [1.0, 3.0, 10.0],
}


def search_space(engine: str) -> dict:
    sp = dict(SEARCH_SPACE)
    if engine != "lightgbm":                # у запасного движка веса классов сильно замедляют обучение
        sp["scale_pos_weight"] = [1.0]
    return sp


def make_model(params: dict, engine: str, seed: int, n_estimators: int | None = None):
    p = dict(params)
    for k in ("num_leaves", "min_child_samples", "n_estimators", "subsample_freq", "max_depth"):
        if k in p:                          # после pandas целые приходят как float
            p[k] = int(p[k])
    if n_estimators is not None:
        p["n_estimators"] = n_estimators
    if engine == "lightgbm":
        p.setdefault("subsample_freq", 1)           # без subsample_freq>0 параметр subsample игнорируется
        return lgb.LGBMClassifier(**p, metric="auc", random_state=seed, deterministic=True,
                                  force_row_wise=True, n_jobs=1, verbose=-1)
    return HistGradientBoostingClassifier(
        max_iter=p.get("n_estimators", 300), learning_rate=p.get("learning_rate", 0.05),
        max_leaf_nodes=p.get("num_leaves", 31), min_samples_leaf=p.get("min_child_samples", 20),
        l2_regularization=p.get("reg_lambda", 0.0), max_features=p.get("colsample_bytree", 1.0),
        class_weight=(None if float(p.get("scale_pos_weight", 1.0)) == 1.0
                      else {0: 1.0, 1: float(p["scale_pos_weight"])}), early_stopping=False,
        random_state=seed)


def fit_early_stopping(params, engine, seed, Xtr, ytr, Xva, yva, rounds=100, max_estimators=3000):
    """Обучает на train, останавливается по AUC на validation. Возвращает (модель, лучшая итерация)."""
    if engine == "lightgbm":
        m = make_model(params, engine, seed, n_estimators=max_estimators)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)],
              callbacks=[lgb.early_stopping(rounds, first_metric_only=True, verbose=False)])
        best = int(m.best_iteration_ or max_estimators)
        return m, best
    m = make_model(params, engine, seed, n_estimators=min(max_estimators, 600))
    m.fit(Xtr, ytr)
    best, best_auc, since = 1, -1, 0
    for i, pr in enumerate(m.staged_predict_proba(Xva), start=1):
        a = roc_auc_score(yva, pr[:, 1])
        if a > best_auc:
            best, best_auc, since = i, a, 0
        else:
            since += 1
            if since >= rounds:
                break
    m = make_model(params, engine, seed, n_estimators=best).fit(Xtr, ytr)
    return m, best


def timeseries_search(X, y, engine, seed, n_iter=30, n_splits=5, rounds=100, max_estimators=3000):
    """Случайный поиск; оценка — средний AUC по расширяющимся окнам TimeSeriesSplit внутри train
    (валидация всегда «после» обучения). Возвращает таблицу результатов, отсортированную по качеству."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for k, params in enumerate(ParameterSampler(search_space(engine), n_iter=n_iter, random_state=seed)):
        aucs, iters = [], []
        for tr_i, va_i in tscv.split(X):
            ytr, yva = y.iloc[tr_i], y.iloc[va_i]
            if yva.sum() == 0:
                continue
            m, best = fit_early_stopping(params, engine, seed, X.iloc[tr_i], ytr, X.iloc[va_i], yva,
                                         rounds, max_estimators)
            aucs.append(roc_auc_score(yva, m.predict_proba(X.iloc[va_i])[:, 1]))
            iters.append(best)
        rows.append({**params, "cv_gini": 2 * np.mean(aucs) - 1, "cv_gini_std": 2 * np.std(aucs),
                     "mean_best_iter": float(np.mean(iters))})
    return pd.DataFrame(rows).sort_values("cv_gini", ascending=False).reset_index(drop=True)


class PDCalibrator:
    """Обёртка над CalibratedClassifierCV: ограничивает PD снизу (isotonic на малой выборке даёт ровно 0,
    а PD = 0 для компании не бывает)."""

    def __init__(self, cal, floor=0.0005):
        self.cal, self.floor = cal, floor
        self.classes_ = np.array([0, 1])

    def predict_proba(self, X):
        p = np.clip(self.cal.predict_proba(X)[:, 1], self.floor, 1 - self.floor)
        return np.c_[1 - p, p]


def calibrate(model, Xva, yva, method="isotonic", floor=0.0005):
    """Калибровка вероятностей на validation (модель «заморожена»)."""
    if FrozenEstimator is not None:
        cal = CalibratedClassifierCV(FrozenEstimator(model), method=method)
    else:
        cal = CalibratedClassifierCV(model, method=method, cv="prefit")
    return PDCalibrator(cal.fit(Xva, yva), floor)


def cv_gini_baseline(builder_fn, X, y, n_splits=5):
    """Те же окна TimeSeriesSplit для базовой модели (WoE-карта переобучается в каждом окне)."""
    out = []
    for tr_i, va_i in TimeSeriesSplit(n_splits=n_splits).split(X):
        if y.iloc[va_i].sum() == 0:
            continue
        m = builder_fn().fit(X.iloc[tr_i], y.iloc[tr_i])
        out.append(2 * roc_auc_score(y.iloc[va_i], m.predict_proba(X.iloc[va_i])) - 1)
    return float(np.mean(out)), float(np.std(out))

In [ ]:
%%writefile src/metrics.py
"""Метрики: Gini, KS, PSI, полнота при доле проверок, прибыль портфеля, калибровка."""
from __future__ import annotations

import numpy as np
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score


def gini(y, p) -> float:
    return 2 * roc_auc_score(y, p) - 1


def ks_stat(y, p) -> float:
    order = np.argsort(p)
    y = np.asarray(y)[order]
    cum_bad = np.cumsum(y) / y.sum()
    cum_good = np.cumsum(1 - y) / (1 - y).sum()
    return float(np.max(np.abs(cum_bad - cum_good)))


def psi(expected, actual, bins: int = 10) -> float:
    """Population Stability Index; границы — квантили expected (train)."""
    expected = np.asarray(expected, float)
    actual = np.asarray(actual, float)
    cuts = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(cuts) < 3:                       # почти константа — нечего сравнивать
        return 0.0
    cuts[0], cuts[-1] = -np.inf, np.inf
    e = np.histogram(expected, cuts)[0] / len(expected) + 1e-6
    a = np.histogram(actual, cuts)[0] / len(actual) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))


def recall_at_share(y, p, share: float) -> float:
    """Полнота (доля пойманных банкротов) среди доли `share` компаний с наибольшим скором."""
    y = np.asarray(y)
    k = max(1, int(np.ceil(share * len(y))))
    top = np.argsort(-np.asarray(p), kind="stable")[:k]
    return float(y[top].sum() / y.sum())


def lift_at_share(y, p, share: float) -> float:
    y = np.asarray(y)
    k = max(1, int(np.ceil(share * len(y))))
    top = np.argsort(-np.asarray(p), kind="stable")[:k]
    return float(y[top].mean() / y.mean())


def pr_auc(y, p) -> float:
    return float(average_precision_score(y, p))


def brier(y, p) -> float:
    return float(brier_score_loss(y, p))


def ece(y, p, bins: int = 10) -> float:
    """Expected calibration error по квантильным корзинам."""
    y, p = np.asarray(y), np.asarray(p)
    order = np.argsort(p)
    chunks = np.array_split(order, bins)
    return float(sum(len(c) / len(y) * abs(y[c].mean() - p[c].mean()) for c in chunks if len(c)))


def summary(y, p, shares=(0.05, 0.10, 0.20)) -> dict:
    out = {"gini": gini(y, p), "auc": roc_auc_score(y, p), "ks": ks_stat(y, p),
           "pr_auc": pr_auc(y, p), "brier": brier(y, p), "ece": ece(y, p)}
    for s in shares:
        out[f"recall@{int(s * 100)}%"] = recall_at_share(y, p, s)
    return out


def bootstrap_ci(y, p, fn, n: int = 1000, seed: int = 42, alpha: float = 0.05):
    """Стратифицированный бутстреп (банкротов мало — сохраняем их долю)."""
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    vals = []
    for _ in range(n):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        vals.append(fn(y[idx], p[idx]))
    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])
    return float(lo), float(hi)


# ---------- финансовая оценка (вариант 5: решение «выдать / отказать») ----------

def portfolio_profit(y, p, threshold: float, margin: float, lgd: float, ead: float = 1.0) -> dict:
    """Одобряем компании с PD < threshold.
    Доход: margin*EAD с каждой одобренной «хорошей»; потери: LGD*EAD с каждой одобренной обанкротившейся."""
    y, p = np.asarray(y), np.asarray(p)
    ok = p < threshold
    good, bad = int(((y == 0) & ok).sum()), int(((y == 1) & ok).sum())
    income, loss = margin * ead * good, lgd * ead * bad
    return {"threshold": float(threshold), "approved": int(ok.sum()), "approve_rate": float(ok.mean()),
            "good_approved": good, "bad_approved": bad, "income": income, "loss": loss,
            "profit": income - loss, "profit_per_exposure": (income - loss) / (ead * len(y))}


def expected_profit(p, threshold: float, margin: float, lgd: float, ead: float = 1.0) -> float:
    """Ожидаемая прибыль по прогнозным PD (без знания факта): sum[(1-PD)*m - PD*LGD] по одобренным."""
    p = np.asarray(p)
    ok = p < threshold
    return float(np.sum((1 - p[ok]) * margin * ead - p[ok] * lgd * ead))


def best_threshold(p, margin: float, lgd: float, ead: float = 1.0):
    """Порог, максимизирующий ожидаемую прибыль по прогнозным PD (перебор по уникальным значениям PD)."""
    p = np.asarray(p)
    cands = np.unique(np.concatenate([[p.min() - 1e-9], p, [p.max() + 1e-9]]))
    vals = np.array([expected_profit(p, t, margin, lgd, ead) for t in cands])
    i = int(np.argmax(vals))
    return float(cands[i]), float(vals[i])

In [ ]:
%%writefile src/explain.py
"""SHAP-интерпретация. Основной путь — shap.TreeExplainer (LightGBM). Если пакет shap не установлен
(только в среде проверки), используется запасной метод «замещение признака медианой train» —
результаты такого запуска помечаются method='occlusion' и не являются финальными."""
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import shap
    HAS_SHAP = True
except ImportError:
    shap, HAS_SHAP = None, False


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def explain(model, X: pd.DataFrame, background_median: pd.Series):
    """Возвращает (матрица вкладов в log-odds, базовое значение, метод)."""
    if HAS_SHAP:
        ex = shap.TreeExplainer(model)
        sv = ex.shap_values(X)
        if isinstance(sv, list):                 # старые версии shap: [класс 0, класс 1]
            sv = sv[1]
        sv = np.asarray(sv)
        if sv.ndim == 3:                         # (n, features, classes)
            sv = sv[:, :, 1]
        base = ex.expected_value
        base = float(np.ravel(base)[-1])
        return sv, base, "shap"
    full = _logit(model.predict_proba(X)[:, 1])
    sv = np.zeros(X.shape)
    for j, c in enumerate(X.columns):
        Xo = X.copy()
        Xo[c] = background_median[c]
        sv[:, j] = full - _logit(model.predict_proba(Xo)[:, 1])
    return sv, float(np.mean(full - sv.sum(1))), "occlusion"


def mean_abs(sv: np.ndarray, cols) -> pd.Series:
    return pd.Series(np.abs(sv).mean(0), index=cols).sort_values(ascending=False)


def plot_summary(sv, X: pd.DataFrame, path, max_display=15, method="shap"):
    if HAS_SHAP and method == "shap":
        plt.figure(figsize=(8, 5.2))
        shap.summary_plot(sv, X, max_display=max_display, show=False, plot_size=None)
        plt.tight_layout()
        plt.savefig(path, dpi=170, bbox_inches="tight")
        plt.close()
        return
    imp = mean_abs(sv, X.columns).head(max_display)[::-1]
    fig, ax = plt.subplots(figsize=(8, 5.2))
    rng = np.random.default_rng(0)
    for i, c in enumerate(imp.index):
        j = list(X.columns).index(c)
        v = X[c].rank(pct=True).to_numpy()
        ax.scatter(sv[:, j], i + rng.uniform(-0.3, 0.3, len(v)), c=v, cmap="coolwarm", s=6)
    ax.set_yticks(range(len(imp)))
    ax.set_yticklabels(imp.index, fontsize=8)
    ax.axvline(0, color="gray", lw=0.6)
    ax.set_xlabel("вклад признака в log-odds банкротства (запасной метод: замещение медианой)")
    plt.tight_layout()
    plt.savefig(path, dpi=170)
    plt.close()


def top_contrib(sv_row, x_row: pd.Series, k=5) -> pd.DataFrame:
    idx = np.argsort(-np.abs(sv_row))[:k]
    return pd.DataFrame({"feature": x_row.index[idx], "value": x_row.values[idx], "shap": sv_row[idx]})


def plot_cases(cases: list, path):
    """cases: список dict(title, contrib DataFrame, pd). Три горизонтальные панели вкладов."""
    fig, axes = plt.subplots(1, len(cases), figsize=(4.4 * len(cases), 3.6))
    for ax, c in zip(np.atleast_1d(axes), cases):
        t = c["contrib"].iloc[::-1]
        ax.barh([f[:34] for f in t["feature"]], t["shap"], color=["#C0392B" if v > 0 else "#2E86C1" for v in t["shap"]])
        ax.axvline(0, color="gray", lw=0.6)
        ax.set_title(c["title"], fontsize=9)
        ax.tick_params(axis="y", labelsize=7)
        ax.set_xlabel("вклад в log-odds (красный — к банкротству)", fontsize=7)
    plt.tight_layout()
    plt.savefig(path, dpi=170)
    plt.close()

In [ ]:
%%writefile src/predict.py
"""predict(df) — вероятность банкротства (PD) для новых данных в формате исходного data.csv."""
from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
_BUNDLE = None


def load_bundle(path=ROOT / "model.pkl"):
    global _BUNDLE
    if _BUNDLE is None:
        _BUNDLE = joblib.load(path)
    return _BUNDLE


def predict(df: pd.DataFrame, model_path=ROOT / "model.pkl", return_decision: bool = False):
    """df — строки с теми же колонками, что в исходном data.csv (колонка целевой переменной не нужна).
    Возвращает pd.Series с калиброванной PD; при return_decision=True — DataFrame c колонками pd, decision
    (decision='reject', если PD >= порога, максимизирующего ожидаемую прибыль)."""
    b = load_bundle(model_path)
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]
    if "obs_order" not in d.columns:
        d.insert(0, "obs_order", np.arange(len(d)))
    X = b["feature_builder"].transform(d)
    pd_hat = pd.Series(b["calibrator"].predict_proba(X)[:, 1], index=df.index, name="pd")
    if not return_decision:
        return pd_hat
    out = pd_hat.to_frame()
    out["decision"] = np.where(pd_hat >= b["threshold"], "reject", "approve")
    return out


if __name__ == "__main__":
    import sys
    data = pd.read_csv(sys.argv[1] if len(sys.argv) > 1 else ROOT / "data" / "data.csv")
    print(predict(data.head(5), return_decision=True))

In [ ]:
%%writefile src/pipeline.py
"""Четыре этапа задания как функции. Ноутбуки и run_all.py только вызывают их.
Промежуточные результаты — в artifacts/, итог — artifacts/results.json, графики — reports/figures/."""
from __future__ import annotations

import json
import warnings
from functools import lru_cache

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score

from . import explain as ex
from . import metrics as M
from .data import ROOT, audit_features, describe_features, load_config, load_data, time_split
from .features import NEW_FEATURES, FeatureBuilder, feature_rationales
from .train import (calibrate, cv_gini_baseline, fit_early_stopping, make_model, resolve_engine, search_space,
                    timeseries_search)
from .woe import Scorecard

ART = ROOT / "artifacts"
FIG = ROOT / "reports" / "figures"
ART.mkdir(exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
C_BASE, C_BOOST, C_NEUT = "#7F8C8D", "#1F618D", "#C0392B"


def _save_json(name, obj):
    with open(ART / name, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=float)


def _load_json(name):
    with open(ART / name, encoding="utf-8") as f:
        return json.load(f)


def update_results(section: str, obj: dict):
    path = ART / "results.json"
    res = json.load(open(path, encoding="utf-8")) if path.exists() else {}
    res[section] = obj
    with open(path, "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=2, default=float)


@lru_cache(maxsize=1)
def context():
    """Конфиг, данные, разбиение по времени, удаляемые признаки, FeatureBuilder (fit на train)."""
    cfg = load_config()
    t = cfg["data"]["target"]
    df = load_data(cfg)
    tr, va, te = time_split(df, cfg["split"]["train"], cfg["split"]["valid"])
    drop, audit = audit_features(tr, t, cfg["features"]["corr_duplicate_threshold"])
    fb = FeatureBuilder(drop, tuple(cfg["features"]["clip_quantiles"]), t).fit(tr)
    c = dict(cfg=cfg, target=t, df=df, tr=tr, va=va, te=te, drop=drop, audit=audit, fb=fb,
             engine=resolve_engine(cfg["boosting"]["engine"]))
    c["Xtr"], c["Xva"] = fb.transform(tr), fb.transform(va)
    c["ytr"], c["yva"] = tr[t].reset_index(drop=True), va[t].reset_index(drop=True)
    c["Xtr"], c["Xva"] = c["Xtr"].reset_index(drop=True), c["Xva"].reset_index(drop=True)
    return c


def _test(c):
    """Test создаётся ТОЛЬКО здесь — и вызывается только из этапа 4."""
    Xte = c["fb"].transform(c["te"]).reset_index(drop=True)
    return Xte, c["te"][c["target"]].reset_index(drop=True)


# =========================== ЭТАП 1 ===========================
def step1_data():
    c = context()
    cfg, t, df, tr, va, te = c["cfg"], c["target"], c["df"], c["tr"], c["va"], c["te"]
    seed = cfg["seed"]
    desc = describe_features(tr, t)
    desc["status"] = np.where(desc["feature"].isin(c["drop"]), "удалён", "оставлен")
    desc.to_csv(ART / "features_description.csv", index=False)
    c["audit"].to_csv(ART / "removed_features.csv", index=False)
    split = pd.DataFrame({"sample": ["train", "valid", "test (OOT)"], "n": [len(tr), len(va), len(te)],
                          "bankrupt": [int(x[t].sum()) for x in (tr, va, te)],
                          "bankrupt_rate": [float(x[t].mean()) for x in (tr, va, te)],
                          "row_from": [int(x.obs_order.min()) for x in (tr, va, te)],
                          "row_to": [int(x.obs_order.max()) for x in (tr, va, te)]})
    split.to_csv(ART / "split_summary.csv", index=False)

    # демонстрация: что дало бы случайное разбиение (одна и та же простая модель)
    p = dict(learning_rate=0.05, num_leaves=8, min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
             reg_lambda=1.0, scale_pos_weight=1.0)
    m_t = make_model(p, c["engine"], seed, 150).fit(c["Xtr"], c["ytr"])
    g_time = M.gini(c["yva"], m_t.predict_proba(c["Xva"])[:, 1])
    # случайное разбиение — только на строках train+valid (test до этапа 4 не трогаем)
    pool = pd.concat([tr, va])
    Xp = c["fb"].transform(pool).reset_index(drop=True)
    yp = pool[t].reset_index(drop=True)
    idx = np.random.default_rng(seed).permutation(len(pool))
    a_, b_ = len(tr), len(pool)
    m_r = make_model(p, c["engine"], seed, 150).fit(Xp.iloc[idx[:a_]], yp.iloc[idx[:a_]])
    g_rand = M.gini(yp.iloc[idx[a_:b_]], m_r.predict_proba(Xp.iloc[idx[a_:b_]])[:, 1])
    demo = {"gini_valid_time_split": g_time, "gini_valid_random_split": g_rand}

    # динамика доли банкротств по порядку строк
    blocks = np.array_split(np.arange(len(df)), 10)
    rate = [float(df[t].iloc[b_].mean()) for b_ in blocks]
    fig, ax = plt.subplots(figsize=(5.6, 2.8))
    ax.bar(range(10), rate, color=C_BOOST)
    ax.axvline(6.5, color=C_NEUT, ls="--", lw=1); ax.axvline(8.0, color=C_NEUT, ls="--", lw=1)
    ax.set_xticks(range(10)); ax.set_xticklabels([f"{i + 1}" for i in range(10)])
    ax.set_xlabel("десятая часть выборки (по порядку строк)"); ax.set_ylabel("доля банкротств")
    ax.set_title("Доля банкротств по порядку строк", fontsize=8.5)
    for x0, x1, lab in ((-.5, 6.5, "train"), (6.5, 8.0, "valid"), (8.0, 9.5, "test")):
        ax.text((x0 + x1) / 2, max(rate) * 1.03, lab, ha="center", fontsize=8, color=C_NEUT)
    plt.tight_layout(); plt.savefig(FIG / "fig1_drift.png", dpi=170); plt.close()

    out = {"n_rows": len(df), "n_raw_features": int(len(df.columns) - 2), "bankrupt_total": int(df[t].sum()),
           "split": split.to_dict("records"), "n_removed": int(len(c["drop"])),
           "removed_by_reason": c["audit"]["reason"].value_counts().to_dict(),
           "n_features_after_audit": int(len(c["fb"].raw_cols_)), "n_new_features": len(NEW_FEATURES),
           "block_rates": rate, "split_demo": demo, "missing_total": int(df.isna().sum().sum()),
           "max_univariate_gini": float(desc.gini_univariate.max()), "engine": c["engine"]}
    update_results("step1", out)
    return out


# =========================== ЭТАП 2 ===========================
def _scorecard_factory(cfg, C):
    b = cfg["baseline"]
    return lambda: Scorecard(b["max_bins"], b["min_bin_share"], b["min_bin_bads"], b["iv_min"], b["corr_max"],
                             b.get("max_features", 15), C, b["scorecard"]["base_score"], b["scorecard"]["base_odds"],
                             b["scorecard"]["pdo"], cfg["seed"])


def step2_baseline():
    c = context()
    cfg, Xtr, Xva, ytr, yva = c["cfg"], c["Xtr"], c["Xva"], c["ytr"], c["yva"]
    grid = []
    for C in cfg["baseline"]["c_grid"]:
        mu, sd = cv_gini_baseline(_scorecard_factory(cfg, C), Xtr, ytr)
        grid.append({"C": C, "cv_gini": mu, "cv_gini_std": sd})
    grid = pd.DataFrame(grid)
    bestC = float(grid.sort_values("cv_gini", ascending=False).iloc[0]["C"])
    sc = _scorecard_factory(cfg, bestC)().fit(Xtr, ytr)
    p_tr, p_va = sc.predict_proba(Xtr), sc.predict_proba(Xva)
    cal = calibrate_prefit_scorecard(sc, Xva, yva)
    sh = cfg["finance"]["check_shares"]
    res = {"best_C": bestC, "n_features": len(sc.features_), "features": sc.features_,
           "cv_gini": float(grid.cv_gini.max()), "cv_gini_std": float(grid.sort_values("cv_gini").iloc[-1].cv_gini_std),
           "train": M.summary(ytr, p_tr, sh), "valid": M.summary(yva, p_va, sh),
           "valid_calibrated": M.summary(yva, cal.predict_proba(Xva)[:, 1], sh),
           "valid_gini_ci": M.bootstrap_ci(yva, p_va, M.gini, cfg["bootstrap"]["n"], cfg["seed"]),
           "sign_ok": bool((sc.lr_.coef_ < 0).all()), "grid": grid.to_dict("records")}
    pts = sc.points_table()
    pts.to_csv(ART / "scorecard_points.csv", index=False)
    pd.DataFrame({"feature": sc.iv_series_.index, "iv": sc.iv_series_.values,
                  "selected": sc.iv_series_.index.isin(sc.features_)}).to_csv(ART / "iv_table.csv", index=False)
    joblib.dump({"scorecard": sc, "calibrator": cal}, ART / "baseline.pkl")
    update_results("step2", res)
    return res


class ScorecardCalibrator:
    """Isotonic-калибровка выхода скоринговой карты (обучается на validation)."""
    def __init__(self, sc, Xva, yva):
        from sklearn.isotonic import IsotonicRegression
        self.sc = sc
        self.iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(sc.predict_proba(Xva), yva)

    def predict_proba(self, X):
        p = self.iso.predict(self.sc.predict_proba(X))
        return np.c_[1 - p, p]


def calibrate_prefit_scorecard(sc, Xva, yva):
    return ScorecardCalibrator(sc, Xva, yva)


# =========================== ЭТАП 3 ===========================
def _write_final_params(final: dict):
    """Обновляет только блок boosting.final_params в config.yaml, сохраняя комментарии."""
    import re
    path = ROOT / "config.yaml"
    txt = path.read_text(encoding="utf-8")
    lines = ["  final_params:"] + [f"    {k}: {v}" for k, v in
                                   {k: (float(v) if isinstance(v, (float, np.floating)) else int(v)) for k, v in final.items()}.items()]
    new = re.sub(r"  final_params:\n(?:    .*\n)+", "\n".join(lines) + "\n", txt)
    path.write_text(new, encoding="utf-8")


def _reliability(ax, y, p, label, color, bins=6):
    fy, fx = calibration_curve(y, p, n_bins=bins, strategy="quantile")
    ax.plot(fx, fy, "o-", color=color, label=label, ms=4)


def step3_boosting():
    c = context()
    cfg, Xtr, Xva, ytr, yva, eng = c["cfg"], c["Xtr"], c["Xva"], c["ytr"], c["yva"], c["engine"]
    seed, bc = cfg["seed"], cfg["boosting"]
    n_iter = bc["search"]["n_iter"] if eng == "lightgbm" else min(bc["search"]["n_iter"], 8)
    max_est = bc["max_estimators"] if eng == "lightgbm" else 400
    rounds = bc["early_stopping_rounds"]
    search = timeseries_search(Xtr, ytr, eng, seed, n_iter, bc["search"]["n_splits"], rounds, max_est)
    search.to_csv(ART / "search_results.csv", index=False)
    keys = list(search_space(eng))
    best = {k: (int(v) if k in ("num_leaves", "min_child_samples") else float(v)) for k, v in search.iloc[0][keys].items()}
    model, n_best = fit_early_stopping(best, eng, seed, Xtr, ytr, Xva, yva, rounds, max_est)
    final = {**best, "n_estimators": int(n_best), "subsample_freq": 1, "reg_alpha": 0.0}
    if eng != "lightgbm":
        final = {k: v for k, v in final.items() if k not in ("subsample_freq", "reg_alpha")}

    p_raw = model.predict_proba(Xva)[:, 1]
    cal = calibrate(model, Xva, yva, cfg["calibration"]["method"], cfg["calibration"]["pd_floor"])
    p_cal = cal.predict_proba(Xva)[:, 1]
    sh = cfg["finance"]["check_shares"]

    # вклад новых признаков: та же конфигурация на «сырых» признаках
    raw_cols = c["fb"].raw_cols_
    m_raw, n_raw = fit_early_stopping(best, eng, seed, Xtr[raw_cols], ytr, Xva[raw_cols], yva, rounds, max_est)
    g_rawfeat = M.gini(yva, m_raw.predict_proba(Xva[raw_cols])[:, 1])

    base = _load_json("results.json")["step2"]
    table = pd.DataFrame([
        {"model": "Скоринговая карта (WoE + логрег)", "cv_gini_train": base["cv_gini"], **{k: base["valid"][k] for k in ("gini", "ks", "pr_auc", "brier", "recall@10%")}},
        {"model": "Бустинг (сырые вероятности)", "cv_gini_train": float(search.iloc[0].cv_gini), **{k: M.summary(yva, p_raw, sh)[k] for k in ("gini", "ks", "pr_auc", "brier", "recall@10%")}},
        {"model": "Бустинг + isotonic-калибровка", "cv_gini_train": float(search.iloc[0].cv_gini), **{k: M.summary(yva, p_cal, sh)[k] for k in ("gini", "ks", "pr_auc", "brier", "recall@10%")}},
    ])
    table.to_csv(ART / "comparison_valid.csv", index=False)
    ci_b = M.bootstrap_ci(yva, p_raw, M.gini, cfg["bootstrap"]["n"], seed)

    # калибровочная кривая на validation (по сути in-sample для isotonic — честная кривая на test в этапе 4)
    fig, ax = plt.subplots(figsize=(3.4, 3.2))
    ax.plot([0, max(p_raw.max(), .1)], [0, max(p_raw.max(), .1)], ":", color="gray")
    _reliability(ax, yva, p_raw, "до калибровки", C_BASE)
    _reliability(ax, yva, p_cal, "isotonic", C_BOOST)
    ax.set_xlabel("прогнозная PD"); ax.set_ylabel("доля банкротств"); ax.legend(fontsize=7)
    ax.set_title("Калибровка, validation", fontsize=8.5)
    plt.tight_layout(); plt.savefig(FIG / "fig_calibration_valid.png", dpi=170); plt.close()

    joblib.dump({"model": model, "calibrator": cal, "params": final, "engine": eng}, ART / "boosting.pkl")
    _write_final_params(final)

    res = {"engine": eng, "final_params": final, "best_iteration": int(n_best), "n_search": int(len(search)),
           "search_best_cv_gini": float(search.iloc[0].cv_gini), "search_best_cv_std": float(search.iloc[0].cv_gini_std),
           "search_median_cv_gini": float(search.cv_gini.median()),
           "valid_raw": M.summary(yva, p_raw, sh), "valid_calibrated": M.summary(yva, p_cal, sh),
           "valid_gini_ci": ci_b, "gini_valid_raw_features_only": g_rawfeat, "n_features_model": int(Xtr.shape[1]),
           "comparison": table.to_dict("records")}
    update_results("step3", res)
    return res


# =========================== ЭТАП 4 ===========================
def _psi_table(ctx, model, sv_train, X_tr, X_va, X_te, p_tr, p_va, p_te, base_scores):
    bc = ctx["cfg"]["stability"]
    top = ex.mean_abs(sv_train, X_tr.columns).head(bc["top_features"]).index.tolist()
    rows = []
    for name, a, b, d in [("score (бустинг)", p_tr, p_va, p_te)] + [(f, X_tr[f].values, X_va[f].values, X_te[f].values) for f in top] + \
                         [("score (скоринговая карта)",) + base_scores]:
        rows.append({"item": name, "psi_train_valid": M.psi(a, b, bc["bins"]), "psi_train_test": M.psi(a, d, bc["bins"]),
                     "train_median": float(np.median(a)), "test_median": float(np.median(d)),
                     "median_shift_pct": float((np.median(d) - np.median(a)) / (abs(np.median(a)) + 1e-12) * 100)})
    t = pd.DataFrame(rows)
    t["flag"] = np.where(t.psi_train_test > bc["psi_threshold"], "НЕСТАБИЛЕН (>0,25)",
                         np.where(t.psi_train_test > bc["psi_warn"], "умеренный сдвиг", "стабилен"))
    return t, top


def step4_evaluation():
    c = context()
    cfg, t, eng = c["cfg"], c["target"], c["engine"]
    Xtr, Xva, ytr, yva = c["Xtr"], c["Xva"], c["ytr"], c["yva"]
    Xte, yte = _test(c)                                 # ← единственное обращение к test
    bst = joblib.load(ART / "boosting.pkl")
    model, cal = bst["model"], bst["calibrator"]
    base = joblib.load(ART / "baseline.pkl")
    sc, cal_b = base["scorecard"], base["calibrator"]
    fin, sh, nb, seed = cfg["finance"], cfg["finance"]["check_shares"], cfg["bootstrap"]["n"], cfg["seed"]

    p_raw_te, p_cal_te = model.predict_proba(Xte)[:, 1], cal.predict_proba(Xte)[:, 1]
    p_sc_te, p_scc_te = sc.predict_proba(Xte), cal_b.predict_proba(Xte)[:, 1]
    p_cal_va, p_scc_va = cal.predict_proba(Xva)[:, 1], cal_b.predict_proba(Xva)[:, 1]
    p_raw_tr, p_raw_va = model.predict_proba(Xtr)[:, 1], model.predict_proba(Xva)[:, 1]

    # ---- 1. метрики варианта на out-of-time ----
    def block(p, pc):
        s = M.summary(yte, p, sh)
        s["brier_calibrated"], s["ece_calibrated"] = M.brier(yte, pc), M.ece(yte, pc)
        s["gini_ci"] = M.bootstrap_ci(yte, p, M.gini, nb, seed)
        s["ks_ci"] = M.bootstrap_ci(yte, p, M.ks_stat, nb, seed)
        for sv in sh:
            s[f"recall@{int(sv * 100)}%_ci"] = M.bootstrap_ci(yte, p, lambda a, b, sv=sv: M.recall_at_share(a, b, sv), nb, seed)
            s[f"lift@{int(sv * 100)}%"] = M.lift_at_share(yte, p, sv)
        return s
    test_boost, test_base = block(p_raw_te, p_cal_te), block(p_sc_te, p_scc_te)
    # разность Gini бустинг − база с бутстреп-ИнтервалоМ (пары на одних и тех же наблюдениях)
    rng = np.random.default_rng(seed)
    pos, neg = np.where(yte == 1)[0], np.where(yte == 0)[0]
    diffs = []
    for _ in range(nb):
        i = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        diffs.append(M.gini(yte.iloc[i], p_raw_te[i]) - M.gini(yte.iloc[i], p_sc_te[i]))
    gini_diff = {"mean": float(np.mean(diffs)), "ci": [float(np.quantile(diffs, .025)), float(np.quantile(diffs, .975))],
                 "share_boost_better": float(np.mean(np.array(diffs) > 0))}

    # ---- 2. деньги ----
    lgd, m0, ead = fin["lgd"], fin["margin"], fin["ead"]
    thr, exp_val = M.best_threshold(p_cal_va, m0, lgd, ead)           # порог выбран на validation по ожидаемой прибыли
    thr_b, _ = M.best_threshold(p_scc_va, m0, lgd, ead)
    all_ok = M.portfolio_profit(yte, p_cal_te, 2.0, m0, lgd, ead)
    boost_pf = M.portfolio_profit(yte, p_cal_te, thr, m0, lgd, ead)
    base_pf = M.portfolio_profit(yte, p_scc_te, thr_b, m0, lgd, ead)
    oracle = M.portfolio_profit(yte, yte.astype(float).to_numpy(), 0.5, m0, lgd, ead)
    # реализованная прибыль на сетке порогов (для графика): отказ топ-k% по скору
    shares = np.linspace(0, 0.30, 61)
    def profit_by_reject_share(p):
        order = np.argsort(-p, kind="stable"); out = []
        for s in shares:
            k = int(round(s * len(p))); keep = np.ones(len(p), bool); keep[order[:k]] = False
            yk = yte.to_numpy()[keep]
            out.append((m0 * (yk == 0).sum() - lgd * (yk == 1).sum()) * ead / (ead * len(p)))
        return np.array(out)
    pr_boost, pr_base = profit_by_reject_share(p_raw_te), profit_by_reject_share(p_sc_te)
    sens = []
    for m in fin["margin_sensitivity"]:
        tm, _ = M.best_threshold(p_cal_va, m, lgd, ead)
        a, b_, o = M.portfolio_profit(yte, p_cal_te, 2.0, m, lgd, ead), M.portfolio_profit(yte, p_cal_te, tm, m, lgd, ead), None
        tb, _ = M.best_threshold(p_scc_va, m, lgd, ead)
        bb = M.portfolio_profit(yte, p_scc_te, tb, m, lgd, ead)
        sens.append({"margin": m, "threshold_pd": tm, "reject_rate": 1 - b_["approve_rate"],
                     "profit_all_approved": a["profit_per_exposure"], "profit_boosting": b_["profit_per_exposure"],
                     "profit_baseline": bb["profit_per_exposure"], "gain_vs_all": b_["profit_per_exposure"] - a["profit_per_exposure"]})
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))
    ax = axes[0]
    ax.plot(shares * 100, pr_boost * 100, color=C_BOOST, label="бустинг"); ax.plot(shares * 100, pr_base * 100, color=C_BASE, label="скоринговая карта")
    ax.axhline(all_ok["profit_per_exposure"] * 100, color="gray", ls=":", label="одобрить всех")
    ax.set_xlabel("% компаний с отказом (наибольший скор)"); ax.set_ylabel("прибыль, % от экспозиции"); ax.legend(fontsize=7)
    ax.set_title("Прибыль на test от доли отказов", fontsize=8.5)
    ax = axes[1]
    ks = np.linspace(0, 1, 101); gb = [M.recall_at_share(yte, p_raw_te, max(s, 1 / len(yte))) for s in ks]
    gs = [M.recall_at_share(yte, p_sc_te, max(s, 1 / len(yte))) for s in ks]
    ax.plot(ks * 100, np.array(gb) * 100, color=C_BOOST, label="бустинг"); ax.plot(ks * 100, np.array(gs) * 100, color=C_BASE, label="скоринговая карта")
    ax.plot([0, 100], [0, 100], ":", color="gray"); ax.set_xlim(0, 40); ax.set_ylim(0, 105)
    ax.set_xlabel("% компаний, отобранных на проверку"); ax.set_ylabel("полнота, % банкротов"); ax.legend(fontsize=7)
    ax.set_title("Кривая полноты (cumulative gains), test", fontsize=8.5)
    plt.tight_layout(); plt.savefig(FIG / "fig_finance.png", dpi=170); plt.close()

    # сравнение способов калибровки на test (калибраторы обучены на validation)
    from sklearn.linear_model import LogisticRegression
    lg = lambda p_: np.log(np.clip(p_, 1e-6, 1 - 1e-6) / (1 - np.clip(p_, 1e-6, 1 - 1e-6))).reshape(-1, 1)
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(lg(p_raw_va), yva)
    p_sig_te = platt.predict_proba(lg(p_raw_te))[:, 1]
    calib_cmp = {"raw": {"brier": M.brier(yte, p_raw_te), "ece": M.ece(yte, p_raw_te), "mean_pd": float(p_raw_te.mean())},
                 "isotonic": {"brier": M.brier(yte, p_cal_te), "ece": M.ece(yte, p_cal_te), "mean_pd": float(p_cal_te.mean())},
                 "sigmoid": {"brier": M.brier(yte, p_sig_te), "ece": M.ece(yte, p_sig_te), "mean_pd": float(p_sig_te.mean())},
                 "actual_rate": float(yte.mean())}

    # калибровка на test (честная)
    fig, ax = plt.subplots(figsize=(3.4, 3.2))
    mx = max(p_raw_te.max(), p_cal_te.max(), .1)
    ax.plot([0, mx], [0, mx], ":", color="gray")
    _reliability(ax, yte, p_raw_te, "бустинг, до калибровки", C_BASE, 5); _reliability(ax, yte, p_cal_te, "бустинг, isotonic", C_BOOST, 5)
    ax.set_xlabel("прогнозная PD"); ax.set_ylabel("доля банкротств"); ax.legend(fontsize=7); ax.set_title("Калибровка, test (OOT)", fontsize=8.5)
    plt.tight_layout(); plt.savefig(FIG / "fig_calibration_test.png", dpi=170); plt.close()

    # ---- 3. SHAP ----
    med = Xtr.median()
    samp = Xtr.sample(min(1500, len(Xtr)), random_state=seed)
    sv_tr, _, method_tr = ex.explain(model, samp, med)
    sv_te, base_val, method = ex.explain(model, Xte, med)
    ex.plot_summary(sv_te, Xte, FIG / "fig_shap_summary.png", 15, method)
    imp = ex.mean_abs(sv_te, Xte.columns)
    from scipy.stats import spearmanr
    shap_dir = {}
    for f_ in imp.head(10).index:
        j = list(Xte.columns).index(f_)
        r_ = spearmanr(Xte[f_], sv_te[:, j])[0]
        shap_dir[f_] = float(0.0 if np.isnan(r_) else r_)           # >0: рост значения признака повышает риск
    pos_i, neg_i = np.where(yte == 1)[0], np.where(yte == 0)[0]
    pick = {"A": pos_i[np.argmax(p_raw_te[pos_i])], "B": neg_i[np.argmax(p_raw_te[neg_i])], "C": pos_i[np.argmin(p_raw_te[pos_i])]}
    titles = {"A": "A: отказ, банкрот (верно)", "B": "B: отказ, не банкрот (ложная тревога)", "C": "C: банкрот, пропущен"}
    cases, cases_out = [], []
    for k, i in pick.items():
        tc = ex.top_contrib(sv_te[i], Xte.iloc[i], 5)
        cases.append({"title": f"{titles[k]}\nPD={p_cal_te[i]:.1%}", "contrib": tc})
        cases_out.append({"case": k, "title": titles[k], "row_in_test": int(i), "target": int(yte.iloc[i]), "pd_calibrated": float(p_cal_te[i]),
                          "score_pct_rank": float((p_raw_te < p_raw_te[i]).mean()), "decision": "отказ" if p_cal_te[i] >= thr else "одобрение",
                          "top": tc.to_dict("records")})
    ex.plot_cases(cases, FIG / "fig_shap_cases.png")

    # ---- 4. PSI ----
    psi_tab, top = _psi_table(c, model, sv_tr, Xtr, Xva, Xte, p_raw_tr, p_raw_va, p_raw_te,
                              (sc.predict_proba(Xtr), sc.predict_proba(Xva), sc.predict_proba(Xte)))
    psi_tab.to_csv(ART / "psi.csv", index=False)
    fig, ax = plt.subplots(figsize=(6.2, 2.7))
    yy = np.arange(len(psi_tab))[::-1]
    ax.barh(yy + .18, psi_tab.psi_train_valid, .34, color=C_BASE, label="train → valid")
    ax.barh(yy - .18, psi_tab.psi_train_test, .34, color=C_BOOST, label="train → test")
    ax.axvline(cfg["stability"]["psi_threshold"], color=C_NEUT, ls="--", lw=1); ax.axvline(cfg["stability"]["psi_warn"], color="orange", ls=":", lw=1)
    ax.set_yticks(yy); ax.set_yticklabels([s[:38] for s in psi_tab["item"]], fontsize=7.5); ax.set_xlabel("PSI"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(FIG / "fig_psi.png", dpi=170); plt.close()

    # ---- 5. model.pkl + проверка predict(df) ----
    bundle = {"feature_builder": c["fb"], "model": model, "calibrator": cal, "threshold": thr, "engine": eng,
              "params": bst["params"], "features": list(Xtr.columns)}
    joblib.dump(bundle, ROOT / "model.pkl")
    from . import predict as P
    P._BUNDLE = None
    chk = P.predict(c["te"].drop(columns=[t, "obs_order"]).iloc[:50])
    pred_ok = bool(np.allclose(chk.values, p_cal_te[:50], atol=1e-9))

    res = {"engine": eng, "shap_method": method, "boost": test_boost, "base": test_base, "gini_diff": gini_diff,
           "finance": {"margin": m0, "lgd": lgd, "threshold_pd": thr, "threshold_pd_baseline": thr_b,
                       "expected_profit_valid": exp_val, "calibration_compare": calib_cmp, "all_approved": all_ok, "boosting": boost_pf,
                       "baseline": base_pf, "oracle": oracle, "sensitivity": sens,
                       "money_example_ead": fin["ead_money_example"]},
           "shap_top": imp.head(10).to_dict(), "shap_direction": shap_dir, "shap_top_new": {k: float(imp[k]) for k in NEW_FEATURES if k in imp.index}, "cases": cases_out, "psi": psi_tab.to_dict("records"),
           "top_features_stability": top, "predict_check_ok": pred_ok,
           "test_n": int(len(yte)), "test_bankrupt": int(yte.sum())}
    update_results("step4", res)
    return res


# =========================== вспомогательное для ноутбуков ===========================
_STEPS = {"step1": step1_data, "step2": step2_baseline, "step3": step3_boosting, "step4": step4_evaluation}


def ensure(step: str) -> dict:
    """Результат этапа из artifacts/results.json; если этапа ещё не было — запускает его."""
    path = ART / "results.json"
    if path.exists():
        res = json.load(open(path, encoding="utf-8"))
        if step in res:
            return res[step]
    for s in ("step1", "step2", "step3", "step4"):      # этапы зависят друг от друга — идём по порядку
        have = path.exists() and s in json.load(open(path, encoding="utf-8"))
        if not have:
            _STEPS[s]()
        if s == step:
            break
    return json.load(open(path, encoding="utf-8"))[step]


def show_fig(name: str):
    from IPython.display import Image, display
    display(Image(filename=str(FIG / name)))

In [ ]:
%%writefile src/report.py
"""Генерация отчёта (docx, 5–7 стр.) из artifacts/results.json и графиков. Все числа берутся из результатов запуска."""
from __future__ import annotations

import json

import pandas as pd
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Pt, RGBColor

from .data import ROOT
from .features import NEW_FEATURES, feature_rationales

ART, FIG = ROOT / "artifacts", ROOT / "reports" / "figures"
FONT = "Times New Roman"


def f(x, d=3):
    return f"{x:.{d}f}".replace(".", ",")


def pc(x, d=1):
    return f"{x * 100:.{d}f}".replace(".", ",") + "%"


def _shade(cell, color):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), color)
    tcPr.append(shd)


def _cell_margins(table, top=20, bottom=20, left=60, right=60):
    tblPr = table._tbl.tblPr
    m = OxmlElement("w:tblCellMar")
    for k, v in (("top", top), ("left", left), ("bottom", bottom), ("right", right)):
        e = OxmlElement(f"w:{k}"); e.set(qn("w:w"), str(v)); e.set(qn("w:type"), "dxa"); m.append(e)
    tblPr.append(m)


class Doc:
    def __init__(self):
        self.d = Document()
        s = self.d.sections[0]
        s.page_width, s.page_height = Cm(21), Cm(29.7)
        s.left_margin = s.right_margin = Cm(2.0)
        s.top_margin, s.bottom_margin = Cm(1.8), Cm(1.8)
        st = self.d.styles["Normal"]
        st.font.name, st.font.size = FONT, Pt(10)
        st.element.rPr.rFonts.set(qn("w:eastAsia"), FONT)
        st.paragraph_format.space_after = Pt(3)
        st.paragraph_format.line_spacing = 1.05
        for name, size in (("Heading 1", 12.5), ("Heading 2", 11)):
            h = self.d.styles[name]
            h.font.name, h.font.size, h.font.bold = FONT, Pt(size), True
            h.font.color.rgb = RGBColor(0x1F, 0x3A, 0x5F)
            h.element.rPr.rFonts.set(qn("w:eastAsia"), FONT)
            h.paragraph_format.space_before, h.paragraph_format.space_after = Pt(8), Pt(3)

    def h(self, t, lvl=1):
        self.d.add_heading(t, lvl)

    def p(self, text, bold_prefix=None, size=None, italic=False, align=None):
        para = self.d.add_paragraph()
        if bold_prefix:
            r = para.add_run(bold_prefix); r.bold = True
            if size: r.font.size = Pt(size)
        r = para.add_run(text); r.italic = italic
        if size: r.font.size = Pt(size)
        if align == "j":
            para.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
        return para

    def bullets(self, items, size=None):
        for it in items:
            para = self.d.add_paragraph(style="List Bullet")
            para.paragraph_format.space_after = Pt(1)
            if isinstance(it, tuple):
                r = para.add_run(it[0]); r.bold = True
                r2 = para.add_run(it[1])
                if size: r.font.size = r2.font.size = Pt(size)
            else:
                r = para.add_run(it)
                if size: r.font.size = Pt(size)

    def table(self, header, rows, widths=None, size=8.5, align_right_from=1, bold_last=False):
        t = self.d.add_table(rows=1, cols=len(header))
        t.style = "Table Grid"; t.alignment = WD_TABLE_ALIGNMENT.CENTER
        _cell_margins(t)
        for i, h_ in enumerate(header):
            c = t.rows[0].cells[i]; c.text = ""
            r = c.paragraphs[0].add_run(str(h_)); r.bold = True; r.font.size = Pt(size)
            _shade(c, "DCE6F1")
        for ri, row in enumerate(rows):
            cells = t.add_row().cells
            for i, v in enumerate(row):
                cells[i].text = ""
                para = cells[i].paragraphs[0]
                para.paragraph_format.space_after = Pt(0)
                r = para.add_run(str(v)); r.font.size = Pt(size)
                if bold_last and ri == len(rows) - 1: r.bold = True
                if i >= align_right_from and align_right_from >= 0:
                    para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        if widths:
            t.autofit = False
            lay = OxmlElement("w:tblLayout"); lay.set(qn("w:type"), "fixed"); t._tbl.tblPr.append(lay)
            for i, w in enumerate(widths):
                t.columns[i].width = Cm(w)
            for row in t.rows:
                for i, w in enumerate(widths):
                    row.cells[i].width = Cm(w)
        self.d.add_paragraph().paragraph_format.space_after = Pt(1)
        return t

    def image(self, path, width_cm, caption=None):
        para = self.d.add_paragraph(); para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        para.add_run().add_picture(str(path), width=Cm(width_cm))
        para.paragraph_format.space_after = Pt(0)
        if caption:
            c = self.d.add_paragraph(); c.alignment = WD_ALIGN_PARAGRAPH.CENTER
            r = c.add_run(caption); r.italic = True; r.font.size = Pt(8.5)

    def two_images(self, a, b, wa, wb, caption=None):
        t = self.d.add_table(rows=1, cols=2); t.alignment = WD_TABLE_ALIGNMENT.CENTER
        for cell, img, w in zip(t.rows[0].cells, (a, b), (wa, wb)):
            para = cell.paragraphs[0]; para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            para.add_run().add_picture(str(img), width=Cm(w))
        if caption:
            c = self.d.add_paragraph(); c.alignment = WD_ALIGN_PARAGRAPH.CENTER
            r = c.add_run(caption); r.italic = True; r.font.size = Pt(8.5)


def _short(rat: str) -> str:
    return rat.split(":")[0].strip() if ":" in rat[:90] else rat.split(".")[0]


def build_report(out=None) -> str:
    R = json.load(open(ART / "results.json", encoding="utf-8"))
    s1, s2, s3, s4 = R["step1"], R["step2"], R["step3"], R["step4"]
    eng = s4["engine"]
    prelim = eng != "lightgbm" or s4["shap_method"] != "shap"
    eng_name = "LightGBM" if eng == "lightgbm" else "HistGradientBoosting (sklearn)"
    shap_name = "SHAP (TreeExplainer)" if s4["shap_method"] == "shap" else "вклады признаков методом замещения медианой"
    B, Bs = s4["boost"], s4["base"]
    fin = s4["finance"]
    sp = {r["sample"]: r for r in s1["split"]}
    tr, va, te = sp["train"], sp["valid"], sp["test (OOT)"]
    D = Doc()

    # ---------- титул ----------
    t = D.d.add_paragraph(); t.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = t.add_run("Прогноз банкротства компаний: скоринговая карта и градиентный бустинг"); r.bold = True; r.font.size = Pt(15)
    t = D.d.add_paragraph(); t.alignment = WD_ALIGN_PARAGRAPH.CENTER
    t.add_run("Практическое задание «ML-моделирование в финансах», вариант 5. Данные: Kaggle «Company Bankruptcy Prediction» (Тайвань)").font.size = Pt(9.5)
    if prelim:
        w = D.d.add_paragraph(); w.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r = w.add_run("ПРЕДВАРИТЕЛЬНАЯ ВЕРСИЯ: расчёт выполнен на запасном движке (" + eng_name + "; объяснения — " + shap_name +
                      "). Финальные числа получаются командой python run_all.py при установленных LightGBM и SHAP.")
        r.bold = True; r.font.size = Pt(8.5); r.font.color.rgb = RGBColor(0xC0, 0x39, 0x2B)

    # ---------- 1. постановка ----------
    D.h("1. Постановка задачи")
    D.p(f"Требуется по годовой финансовой отчётности компании оценить вероятность банкротства в течение года (PD) и "
        f"превратить её в решение «одобрить / отказать». Целевая переменная — Bankrupt? (0/1). Ключевые метрики варианта: "
        f"Gini и полнота (доля пойманных банкротов) при фиксированной доле проверяемых компаний; затем качество переводится "
        f"в прибыль портфеля при LGD = 45%. Сравниваются интерпретируемая базовая модель (WoE-скоринговая карта + логистическая "
        f"регрессия) и градиентный бустинг ({eng_name}); решения бустинга объясняются через {shap_name}.", align="j")

    # ---------- 2. данные ----------
    D.h("2. Данные, разбиение по времени, проверка утечки")
    D.p(f"Выборка: {s1['n_rows']} компаний, {s1['n_raw_features']} признаков (коэффициенты отчётности: рентабельность, ликвидность, "
        f"долговая нагрузка, оборачиваемость, денежные потоки, рост), пропусков {s1['missing_total']}; банкротств "
        f"{s1['bankrupt_total']} ({pc(s1['bankrupt_total'] / s1['n_rows'])}). Тип всех признаков — число; доля пропусков и экономический "
        f"смысл каждого признака приведены в artifacts/features_description.csv и ноутбуке 01.", align="j")
    D.p("В датасете нет колонки с датой отчёта (данные Taiwan Economic Journal, 1999–2009, записаны хронологически). "
        "Поэтому порядок строк использован как прокси времени: подтверждение — доля банкротств в целом резко снижается по порядку строк "
        f"({pc(min(s1['block_rates'][:4]), 0)}–{pc(max(s1['block_rates'][:4]), 0)} в первых четырёх десятых частях и "
        f"{pc(min(s1['block_rates'][4:]), 0)}–{pc(max(s1['block_rates'][4:]), 0)} в последних шести, рис. 1). "
        "Это главное ограничение работы: календарные даты недоступны, поэтому «out-of-time» понимается как «более поздние строки».",
        bold_prefix="Дата наблюдения. ", align="j")
    D.table(["Выборка", "Строки (порядок)", "Наблюдений", "Банкротств", "Доля банкротств"],
            [[n, f"{r['row_from']}–{r['row_to']}", r["n"], r["bankrupt"], pc(r["bankrupt_rate"], 2)]
             for n, r in (("train (70%)", tr), ("validation (15%)", va), ("test, out-of-time (15%)", te))],
            widths=[4.2, 3.5, 3, 3, 3.3])
    d = s1["split_demo"]
    D.p(f"Разбиение выполнено по времени: train — первые 70%, validation — следующие 15%, test — последние 15%; test не использовался "
        f"до этапа 4 и открывается один раз в коде. Для сравнения, одна и та же простая модель даёт Gini {f(d['gini_valid_time_split'])} "
        f"на validation при разбиении по времени и {f(d['gini_valid_random_split'])} при случайном: при дрейфе доли банкротов оценка на "
        f"перемешанных периодах некорректна, поэтому случайное разбиение не применялось. Все статистики (границы винзоризации, пороги "
        f"флага стресса, WoE-корзины, медианы) считаются только на train.", bold_prefix="Разбиение. ", align="j")
    rem = pd.read_csv(ART / "removed_features.csv")
    const = rem[rem.reason == "constant"].feature.tolist()
    dup = rem[rem.reason == "duplicate"]
    leak = rem[rem.reason == "leak_suspect"]
    D.p(f"Проверена по трём направлениям. (1) По смыслу: все 95 признаков — коэффициенты отчётности за период до банкротства; признаков, "
        f"становящихся известными только после события (суммы просрочки, «будущие» цены), нет. (2) Статистически: максимальный "
        f"одномерный Gini признака на train — {f(s1['max_univariate_gini'])}, подозрительно высоких (> 0,90) значений "
        f"{'нет' if leak.empty else 'есть: ' + ', '.join(leak.feature)}. (3) Избыточность: удалено {len(rem)} признаков — "
        f"константа {', '.join(const) if const else '—'} и {len(dup)} дублей (|ρ Спирмена| ≥ 0,98; оставлен признак с большим Gini на train). "
        f"Каждый удалённый признак перечислен в таблице.", bold_prefix="Утечка данных. ", align="j")
    rows = [[r.feature, "константа" if r.reason == "constant" else ("дубль" if r.reason == "duplicate" else "подозрение на утечку"),
             (r.detail + (f"; оставлен: {r.partner}" if isinstance(r.partner, str) and r.partner else ""))] for r in rem.itertuples()]
    D.table(["Удалённый признак", "Причина", "Обоснование"], rows, widths=[6.2, 2.6, 8.2], size=7.5, align_right_from=-1)
    D.two_images(FIG / "fig1_drift.png", FIG / "fig_calibration_test.png", 8.6, 5.6,
                 "Рис. 1. Дрейф доли банкротств по порядку строк (слева); калибровка на test (справа)")

    # ---------- 3. признаки ----------
    D.h("3. Признаки")
    D.p(f"После аудита оставлено {s1['n_features_after_audit']} исходных признаков; добавлено {s1['n_new_features']} новых "
        f"(финальная матрица — {s3['n_features_model']} столбцов). Каждый признак имеет экономическое обоснование; значения винзоризованы "
        f"по 1%/99% квантилям train, деления защищены сдвигом знаменателя.", align="j")
    rat = feature_rationales()
    sh_new = s4.get("shap_top_new", {})
    D.table(["Признак", "Экономический смысл"],
            [[n, rat[n].split(". ")[0].rstrip(".") + "."] for n in NEW_FEATURES], widths=[4.2, 12.8], size=7.5, align_right_from=-1)
    g_raw, g_all = s3["gini_valid_raw_features_only"], s3["valid_raw"]["gini"]
    D.p(f"Вклад новых признаков: бустинг с теми же параметрами на одних исходных признаках даёт Gini {f(g_raw)} на validation, "
        f"со всеми — {f(g_all)}" + (" (разница в пределах шума: на validation всего " + str(va["bankrupt"]) + " банкротов). " if abs(g_all - g_raw) < 0.02 else ". ") +
        (f"В топ-10 по важности SHAP на test попали новые признаки: {', '.join(k for k in s4['shap_top'] if k in NEW_FEATURES) or 'нет'}." ), align="j")

    # ---------- 4. модели ----------
    D.h("4. Базовая модель, бустинг и сравнение")
    D.p(f"Квантильный биннинг до 10 корзин с объединением слабых (≥5% наблюдений и ≥5 банкротов в корзине), WoE со сглаживанием; отбор по IV ≥ 0,05 "
        f"и корреляции WoE < 0,7; логистическая регрессия (C = {s2['best_C']} выбрано по TimeSeriesSplit); в карте {s2['n_features']} признаков, "
        f"знаки коэффициентов экономически корректны ({'да' if s2['sign_ok'] else 'нет'}); шкала: 600 баллов при odds 50:1, PDO = 20.",
        bold_prefix="Базовая модель. ", align="j")
    p = s3["final_params"]
    D.p(f"{eng_name}; гиперпараметры подобраны случайным поиском ({s3['n_search']} конфигураций) с оценкой по TimeSeriesSplit внутри train "
        f"(5 расширяющихся окон, обычный KFold не используется), затем ранняя остановка по AUC на validation (итераций: {s3['best_iteration']}). "
        f"Итог: num_leaves = {p['num_leaves']}, learning_rate = {p['learning_rate']}, min_child_samples = {p['min_child_samples']}, "
        f"colsample = {p['colsample_bytree']}, reg_lambda = {p['reg_lambda']}, scale_pos_weight = {p['scale_pos_weight']}; параметры записаны в config.yaml. "
        f"Вероятности откалиброваны isotonic-регрессией на validation (CalibratedClassifierCV, модель заморожена).",
        bold_prefix="Бустинг. ", align="j")
    c = s3["comparison"]
    D.table(["Модель (validation, 15%)", "CV-Gini (TimeSeriesSplit, train)", "Gini", "KS", "PR-AUC", "Brier", "Полнота@10%"],
            [[x["model"], f(x["cv_gini_train"]), f(x["gini"]), f(x["ks"]), f(x["pr_auc"]), f(x["brier"], 4), pc(x["recall@10%"], 0)] for x in c],
            widths=[5.6, 3.0, 1.6, 1.6, 1.8, 1.7, 2.1], size=8)
    d_cv = s3["search_best_cv_gini"] - s2["cv_gini"]
    D.p(f"Сравнение честное: обе модели оценены на одном и том же validation и по одним и тем же окнам TimeSeriesSplit. По CV на train бустинг "
        f"{'лучше' if d_cv > 0 else 'не лучше'} карты на {f(abs(d_cv))} Gini ({f(s3['search_best_cv_gini'])} против {f(s2['cv_gini'])}, "
        f"стд. по окнам ≈ {f(s3['search_best_cv_std'], 2)}). На validation разница {f(s3['valid_raw']['gini'] - s2['valid']['gini'])} "
        f"в пределах шума (95% ДИ Gini бустинга {f(s3['valid_gini_ci'][0])}–{f(s3['valid_gini_ci'][1])}, банкротов {va['bankrupt']}). "
        f"Калибровка не меняет ранжирование, но улучшает Brier ({f(c[1]['brier'], 4)} → {f(c[2]['brier'], 4)}; на validation оценка для isotonic "
        f"оптимистична, т.к. калибратор обучен на этих же данных — честная проверка на test).", align="j")

    # ---------- 5. финансы ----------
    D.h("5. Out-of-time оценка и финансовый эффект")
    D.p(f"Test (последние 15%, {s4['test_n']} компаний, {s4['test_bankrupt']} банкротств) оценён один раз.", bold_prefix="Метрики на test. ")
    D.table(["Модель (test)", "Gini [95% ДИ]", "KS", "PR-AUC", "Полнота@5%", "Полнота@10%", "Полнота@20%"],
            [[nm, f"{f(x['gini'])} [{f(x['gini_ci'][0], 2)}–{f(x['gini_ci'][1], 2)}]", f(x["ks"]), f(x["pr_auc"]),
              pc(x["recall@5%"], 0), pc(x["recall@10%"], 0), pc(x["recall@20%"], 0)]
             for nm, x in (("Скоринговая карта", Bs), (f"Бустинг ({eng_name})", B))],
            widths=[4.2, 3.6, 1.5, 1.7, 2.0, 2.0, 2.0], size=8)
    gd = s4["gini_diff"]
    verdict = ("бустинг значимо лучше карты" if gd["ci"][0] > 0 else ("карта значимо лучше бустинга" if gd["ci"][1] < 0 else
               "различие между моделями статистически незначимо (интервал включает 0)"))
    D.p(f"Разность Gini (бустинг − карта) на test: {f(gd['mean'])}, 95% ДИ [{f(gd['ci'][0])}; {f(gd['ci'][1])}] — {verdict}. "
        f"При проверке {pc(0.10, 0)} компаний с наибольшим скором бустинг ловит {pc(B['recall@10%'], 0)} банкротов (лифт {f(B['lift@10%'], 1)}), "
        f"карта — {pc(Bs['recall@10%'], 0)}. Малое число банкротов в test ({s4['test_bankrupt']}) делает интервалы широкими; различия в "
        f"1–2 банкрота (один банкрот = {f(100 / s4['test_bankrupt'], 1)} п.п. полноты) не являются доказательством превосходства одной из моделей.", align="j")
    cc = fin["calibration_compare"]
    D.p(f"Brier на test: до калибровки {f(cc['raw']['brier'], 4)}, isotonic {f(cc['isotonic']['brier'], 4)}, sigmoid (Platt) {f(cc['sigmoid']['brier'], 4)}; "
        f"средняя PD {pc(cc['isotonic']['mean_pd'], 2)} при фактической доле {pc(cc['actual_rate'], 2)}. Калибратор ступенчатый ({va['bankrupt']} банкротов на "
        f"validation), поэтому PD дискретны, а для PD введена нижняя граница.", bold_prefix="Калибровка. ", align="j")
    m0, lgd = fin["margin"], fin["lgd"]
    D.p(f"Каждая компания — экспозиция EAD = 1. Одобренная «хорошая» компания приносит маржу {pc(m0, 0)} (допущение: в данных нет сумм и ставок), "
        f"одобренная обанкротившаяся приводит к потере LGD × EAD = {pc(lgd, 0)}. Порог PD выбран на validation по максимуму ожидаемой "
        f"прибыли Σ[(1 − PD)·m − PD·LGD] по калиброванным PD (теоретический ориентир m/(m+LGD) = {pc(m0 / (m0 + lgd), 1)}; на практике "
        f"{pc(fin['threshold_pd'], 0)} из-за ступенчатости калибратора); затем на test посчитана реализованная прибыль.", bold_prefix="Деньги. ", align="j")
    rows = []
    for nm, k in (("Одобрить всех", "all_approved"), ("Скоринговая карта + порог", "baseline"), ("Бустинг + порог", "boosting"), ("Идеал (знаем исход)", "oracle")):
        x = fin[k]
        rows.append([nm, pc(x["approve_rate"], 1), x["bad_approved"], f(x["income"], 2), f(x["loss"], 2), f(x["profit"], 2),
                     pc(x["profit_per_exposure"], 2), f(x["profit_per_exposure"] * 1000, 1)])
    D.table(["Стратегия (test)", "Одобрено", "Одобр. банкротов", "Доход", "Потери", "Прибыль", "Прибыль, % EAD", "млн ₸ на 1000 комп. при EAD 1 млн ₸"], rows,
            widths=[3.8, 1.7, 1.9, 1.5, 1.5, 1.6, 2.0, 3.0], size=8)
    gb, gall = fin["boosting"]["profit_per_exposure"], fin["all_approved"]["profit_per_exposure"]
    sens = fin["sensitivity"]
    D.p(f"Бустинг с порогом даёт {pc(gb, 2)} от экспозиции против {pc(gall, 2)} при одобрении всех (+{f((gb - gall) * 100, 2)} п.п.); карта — "
        f"{pc(fin['baseline']['profit_per_exposure'], 2)}; потолок при полном знании исхода — {pc(fin['oracle']['profit_per_exposure'], 2)}. "
        f"К анализу чувствительности: при марже {pc(sens[0]['margin'], 0)}–{pc(sens[-1]['margin'], 0)} выигрыш против «одобрить всех» составляет "
        f"{f(sens[0]['gain_vs_all'] * 100, 2)}–{f(sens[-1]['gain_vs_all'] * 100, 2)} п.п. EAD. Эффект невелик, потому что банкротов мало "
        f"({pc(s4['test_bankrupt'] / s4['test_n'], 1)}), а эффект сильнее при более высоком LGD или меньшей марже.", align="j")
    D.image(FIG / "fig_finance.png", 15.2, "Рис. 2. Реализованная прибыль и кривая полноты на test")

    # ---------- 6. SHAP ----------
    D.h(f"6. Интерпретация: {('SHAP' if s4['shap_method'] == 'shap' else 'вклады признаков')}")
    top = list(s4["shap_top"].items())[:5]
    dirn = s4.get("shap_direction", {})
    def _d(k):
        v = dirn.get(k)
        return "" if v is None else (" — рост значения повышает риск" if v > 0.2 else (" — рост значения снижает риск" if v < -0.2 else " — связь немонотонна"))
    D.p("Главные драйверы решения модели (среднее |вклад| в log-odds на test): " + "; ".join(f"{k} ({f(v, 2)}{_d(k)})" for k, v in top) +
        ". Направление определено по знаку ранговой корреляции значения признака и его вклада.", align="j")
    D.image(FIG / "fig_shap_summary.png", 11.5, "Рис. 3. Summary plot на test (красный — высокое значение признака)")
    D.p("Для трёх наблюдений из test разобраны вклады пяти сильнейших признаков (вклад > 0 повышает PD).", bold_prefix="Разбор наблюдений. ")
    rows = []
    for cs in s4["cases"]:
        drv = "; ".join(f"{t['feature']} = {t['value']:.3g} ({t['shap']:+.2f})" for t in cs["top"][:4])
        rows.append([cs["title"], pc(cs["pd_calibrated"], 1), ("банкрот" if cs["target"] else "не банкрот") + " / " + cs["decision"], drv])
    D.table(["Наблюдение", "PD", "Факт / решение", "Главные признаки: значение (вклад в log-odds)"], rows, widths=[3.8, 1.3, 2.6, 9.3], size=7.5, align_right_from=-1)
    a, b_, c_ = s4["cases"]
    ta, tb = ([t["feature"] for t in x["top"]] for x in (a, b_))
    common = [x for x in ta[:3] if x in tb[:3]]
    up_c = [t for t in c_["top"] if t["shap"] > 0]
    dn_c = [t for t in c_["top"] if t["shap"] < 0]
    fmt = lambda L: ", ".join(f"{t['feature']} ({t['shap']:+.2f})".replace(".", ",") for t in L)
    D.p(f"Наблюдение A (верный отказ): PD {pc(a['pd_calibrated'], 0)}; решающие признаки — {', '.join(ta[:3])}. "
        f"Наблюдение B — ложная тревога (компания не обанкротилась), PD {pc(b_['pd_calibrated'], 0)}; "
        + (f"главные признаки риска те же ({', '.join(common)}): модель видит у двух компаний похожий профиль, и эти признаки не позволили их различить. " if common else
           f"решающие признаки — {', '.join(tb[:3])}. ")
        + f"Наблюдение C — пропущенный банкрот (PD {pc(c_['pd_calibrated'], 1)}): риск повышали {fmt(up_c)}"
        + (f", но частично их гасили признаки, снижающие PD: {fmt(dn_c)}; итоговая PD осталась ниже порога отказа." if dn_c else "; сигнал оказался слабее порога отказа."), align="j")

    # ---------- 7. стабильность ----------
    D.h("7. Стабильность (PSI)")
    psi = s4["psi"]
    D.table(["Показатель", "PSI train→valid", "PSI train→test", "Медиана train", "Медиана test", "Вывод"],
            [[x["item"], f(x["psi_train_valid"], 3), f(x["psi_train_test"], 3), f"{x['train_median']:.3g}".replace(".", ","), f"{x['test_median']:.3g}".replace(".", ","), x["flag"]] for x in psi],
            widths=[5.2, 2.2, 2.2, 2.2, 2.2, 3.2], size=8)
    bad = [x for x in psi if x["psi_train_test"] > 0.25 and not x["item"].startswith("score")]
    score_psi = psi[0]["psi_train_test"]
    txt = (f"PSI скора бустинга train→test равен {f(score_psi)} — {'скор стабилен' if score_psi < 0.1 else ('умеренный сдвиг' if score_psi < 0.25 else 'скор нестабилен')}. ")
    if bad:
        txt += (f"Нестабильны (PSI > 0,25) {len(bad)} из {len(psi) - 2} топ-признаков: " + ", ".join(f"{x['item']} ({f(x['psi_train_test'], 2)})" for x in bad) + ". ")
        near0 = [x for x in bad if abs(x["train_median"]) < 0.01]
        txt += ("Вероятные причины: (а) дрейф во времени — test это более поздние строки, где доля банкротств упала с "
                f"{pc(tr['bankrupt_rate'], 1)} до {pc(te['bankrupt_rate'], 1)}, т.е. изменился состав компаний и условия; "
                + (f"(б) {len(near0)} из них — коэффициенты со значениями у нуля и тяжёлым правым хвостом (медиана train < 0,01): "
                   "небольшой абсолютный сдвиг перекладывает наблюдения между децильными корзинами и раздувает PSI. " if near0 else "")
                + "Рекомендация: мониторить эти признаки при внедрении, переобучать модель по расписанию и сравнивать вклад нестабильных признаков "
                  "в решение; если их вклад велик, заменить их более стабильными аналогами.")
    else:
        txt += "Среди топ-признаков нестабильных (PSI > 0,25) нет."
    D.p(txt, align="j")

    # ---------- 8. ограничения ----------
    D.h("8. Ограничения и выводы")
    D.bullets([
        ("Нет даты. ", "Порядок строк — прокси времени; календарная валидность разбиения подтверждена лишь дрейфом доли банкротств. Результат нельзя считать полноценным out-of-time на календарных данных."),
        ("Мало событий. ", f"В validation {va['bankrupt']}, в test {te['bankrupt']} банкротств: метрики шумные, различия моделей в пределах доверительных интервалов; для надёжности использован TimeSeriesSplit на train и бутстреп-ДИ."),
        ("Допущения денежной оценки. ", f"Маржа {pc(m0, 0)} и EAD = 1 заданы условно (в данных нет сумм); LGD = 45% из задания; важны не абсолютные цифры, а сравнение стратегий."),
        ("Данные. ", "Тайвань, 1999–2009, обезличенные нормированные коэффициенты; перенос на другой рынок (например, эмитентов KASE) потребует переобучения и проверки стабильности."),
        ("Калибровка. ", "Isotonic по " + str(va["bankrupt"]) + " банкротам на validation даёт ступенчатые PD; для продуктивного использования нужна калибровка на большей выборке."),
    ], size=9)
    cmp_txt = ("Бустинг и скоринговая карта сопоставимы по качеству; карта проще и прозрачнее (баллы по каждой корзине), бустинг даёт запас "
               "по гибкости и объяснимость через SHAP" if abs(gd["mean"]) < 0.03 else
               ("Бустинг превосходит карту" if gd["mean"] > 0 else "Карта превосходит бустинг") + " по Gini на test")
    D.p(f"{cmp_txt}. Работа воспроизводима одной командой (python run_all.py), код вынесен в src/, параметры — в config.yaml, "
        f"модель и функция predict(df) — в model.pkl и src/predict.py.", bold_prefix="Итог. ", align="j")

    out = out or (ROOT / "reports" / "report.docx")
    D.d.save(out)
    return str(out)


if __name__ == "__main__":
    print(build_report())

In [ ]:
# 0.3 Подключаем проект
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/content/bankruptcy-ml")
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.max_colwidth", 400, "display.float_format", "{:.4f}".format)
from src import pipeline as P
from src import train
c = P.context()
print("движок бустинга:", c["engine"], "(должен быть lightgbm)")
import importlib.util
print("shap установлен:", importlib.util.find_spec("shap") is not None)

# Часть 1. 01. Данные, разбиение по времени, аудит признаков (этап 1)
**Вариант 5: прогноз банкротства компаний** — Kaggle «Company Bankruptcy Prediction» (Тайвань, `data.csv`).
Цель — `Bankrupt?` (банкротство в течение года, 0/1). Метрики варианта: Gini и полнота при фиксированной доле проверок.

**Контрольная точка (3–5 предложений).** Вывод приведён в конце ноутбука.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
root = os.getcwd()                              # ищем корень проекта (где лежит config.yaml) — работает и в Colab
while not os.path.exists(os.path.join(root, "config.yaml")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, root); os.chdir(root)
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.float_format", "{:.4f}".format)
from src import pipeline as P
c = P.context()

## 1. Данные: размер, пропуски, цель

In [ ]:
df, t = c["df"], c["target"]
print("строк:", len(df), "| признаков:", df.shape[1] - 2, "| пропусков всего:", int(df.isna().sum().sum()))
print(df[t].value_counts().rename("n").to_frame().assign(share=lambda x: x.n / len(df)))

## 2. Описание каждого признака
Тип, доля пропусков, число уникальных значений, диапазон и одномерный Gini на train. Экономический смысл признаков понятен из названий (все — коэффициенты годовой отчётности: рентабельность, ликвидность, долговая нагрузка, оборачиваемость, денежные потоки, рост). Колонка `status` показывает, оставлен ли признак после аудита.

In [ ]:
r1 = P.ensure("step1")
desc = pd.read_csv("artifacts/features_description.csv")
desc

## 3. Дата наблюдения и разбиение по времени
В датасете **нет колонки с датой**. Данные собраны за 1999–2009 гг. (Taiwan Economic Journal) и записаны в хронологическом порядке, поэтому **порядок строк используется как прокси времени**. Это главное ограничение работы (см. README и отчёт): проверка — доля банкротств монотонно снижается по порядку строк (рисунок ниже), т.е. порядок строк действительно несёт информацию о периоде.

Разбиение: train — первые 70%, validation — следующие 15%, test (out-of-time) — последние 15%. Test в частях 1–3 **не используется**.

In [ ]:
print(pd.read_csv("artifacts/split_summary.csv"))
P.show_fig("fig1_drift.png")

**Случайное разбиение для сравнения.** Одна и та же простая модель, оценённая на validation при разбиении по времени и при случайном разбиении, — демонстрация, почему случайное разбиение недопустимо (оценка получается на «перемешанных» периодах).

In [ ]:
r1["split_demo"]

## 4. Проверка утечки данных и аудит признаков
1. **Признаки «после события»**: просмотрены все 95 названий. Все показатели — коэффициенты отчётности за период ДО банкротства (рентабельность, ликвидность, долг, оборачиваемость, потоки). Признаков вида «сумма просрочки», «статус после» и т.п. нет → утечки по смыслу не найдено.
2. **Статистическая проверка**: одномерный Gini любого признака на train не должен быть подозрительно высоким (порог 0,90).
3. **Константы и дубли**: удаляем признаки, не несущие информации, и «двойников» (|rho| ≥ 0,98 по Спирмену), оставляя более сильного.
4. **Процесс**: все статистики (границы винзоризации, квантили для флага стресса, WoE-бины, пороги) считаются только на train.

Удалённые признаки и причины:

In [ ]:
pd.read_csv("artifacts/removed_features.csv")

In [ ]:
print("максимальный одномерный Gini:", round(r1["max_univariate_gini"], 3))
print("осталось сырых признаков:", r1["n_features_after_audit"])

**Вывод.** В выборке 6819 компаний и 220 банкротов (3,2%). Данные разделены по порядку строк на train (4773), validation (1023) и test (1023); test не использовался до этапа 4. Даты в данных нет, поэтому порядок строк — прокси времени; доля банкротов в целом падает с 4–9% в начале до 1–2% в конце, что это подтверждает. Утечки «после события» в данных нет; удалены 20 признаков: 1 константа и 19 дублей (корреляция не ниже 0,98).

# Часть 2. 02. Признаки и базовая модель (этап 2)
1. ≥ 10 новых признаков (15) с экономическим обоснованием.
2. Классическая скоринговая карта: WoE-биннинг → логистическая регрессия → баллы.
3. Метрики базовой модели на validation — планка для бустинга.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
root = os.getcwd()                              # ищем корень проекта (где лежит config.yaml) — работает и в Colab
while not os.path.exists(os.path.join(root, "config.yaml")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, root); os.chdir(root)
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.float_format", "{:.4f}".format)
from src import pipeline as P
c = P.context()

## 1. Новые признаки и их экономическое обоснование

In [ ]:
from src.features import feature_rationales, NEW_FEATURES
rat = feature_rationales()
Xtr, ytr = c["Xtr"], c["ytr"]
from src import metrics as M
tab = pd.DataFrame({"feature": NEW_FEATURES,
                    "gini_train": [abs(M.gini(ytr, Xtr[f])) for f in NEW_FEATURES],
                    "rationale": [rat[f] for f in NEW_FEATURES]})
print(len(NEW_FEATURES), "новых признаков")
pd.set_option("display.max_colwidth", 400)
tab

## 2. Базовая модель: скоринговая карта
* Квантильный биннинг (до 10 корзин, слияние соседних до ≥5% наблюдений и ≥5 банкротов в корзине), WoE со сглаживанием.
* Отбор: IV ≥ 0,05, жадное удаление коррелированных WoE (|ρ| ≥ 0,7), не более 15 признаков; признаки с «неправильным» знаком коэффициента удаляются.
* Логистическая регрессия (L2); регуляризация `C` выбрана по TimeSeriesSplit на train (не KFold).
* Баллы: base score 600 при odds 50:1, PDO = 20.

In [ ]:
r2 = P.ensure("step2")
print("C =", r2["best_C"], "| признаков в карте:", r2["n_features"], "| знаки коэффициентов корректны:", r2["sign_ok"])
pd.DataFrame(r2["grid"])

In [ ]:
iv = pd.read_csv("artifacts/iv_table.csv"); iv.head(20)

In [ ]:
pts = pd.read_csv("artifacts/scorecard_points.csv")
pts[["feature", "bin", "lo", "hi", "n", "bads", "bad_rate", "woe", "points"]].head(30)

## 3. Метрики на validation — планка для бустинга

In [ ]:
pd.DataFrame({"train": r2["train"], "validation": r2["valid"], "validation (isotonic)": r2["valid_calibrated"]}).T

In [ ]:
print("95% ДИ Gini на validation:", [round(x, 3) for x in r2["valid_gini_ci"]], "(в validation мало банкротов → интервал широкий)")
print("CV-Gini (TimeSeriesSplit на train): %.3f ± %.3f" % (r2["cv_gini"], r2["cv_gini_std"]))

**Вывод.** Скоринговая карта (15 признаков, C = 0,01) даёт Gini 0,969 на validation и 0,795 по TimeSeriesSplit на train. Среди отобранных признаков самые сильные по IV — retained_self_financing, Retained Earnings to Total Assets и interest_drag; все коэффициенты имеют ожидаемый знак. Оценка на validation шумная (13 банкротов, ДИ 0,946–0,987), поэтому ориентир — результат по TimeSeriesSplit. Эти значения — планка для бустинга.

# Часть 3. 03. Градиентный бустинг и подбор параметров (этап 3)
LightGBM; параметры подбираются случайным поиском с оценкой по **TimeSeriesSplit** внутри train (обычный KFold не используется); ранняя остановка по AUC на validation; калибровка isotonic на validation.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
root = os.getcwd()                              # ищем корень проекта (где лежит config.yaml) — работает и в Colab
while not os.path.exists(os.path.join(root, "config.yaml")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, root); os.chdir(root)
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.float_format", "{:.4f}".format)
from src import pipeline as P
c = P.context()

In [ ]:
r3 = P.ensure("step3")
print("движок:", r3["engine"], "| лучшая итерация:", r3["best_iteration"], "| признаков:", r3["n_features_model"])
r3["final_params"]

## Подбор по времени
Таблица поиска (топ-10 по среднему Gini на окнах TimeSeriesSplit). Итоговые параметры записаны в `config.yaml` (раздел `boosting.final_params`).

In [ ]:
pd.read_csv("artifacts/search_results.csv").head(10)

## Сравнение с базовой моделью — на одном и том же validation

In [ ]:
cmp = pd.read_csv("artifacts/comparison_valid.csv"); cmp

In [ ]:
print("95% ДИ Gini бустинга на validation:", [round(x, 3) for x in r3["valid_gini_ci"]])
print("Gini на validation на сырых признаках (без 15 новых):", round(r3["gini_valid_raw_features_only"], 3),
      "| со всеми признаками:", round(r3["valid_raw"]["gini"], 3))

## Калибровка
Isotonic-калибровка (`CalibratedClassifierCV`, модель заморожена, калибратор обучен на validation). Кривая ниже построена на validation, т.е. для isotonic она «в выборке»; честная проверка калибровки — на test в части 4.

In [ ]:
P.show_fig("fig_calibration_valid.png")

**Вывод.** LightGBM (296 деревьев, 16 листьев, шаг 0,02, scale_pos_weight = 3) по TimeSeriesSplit на train даёт Gini 0,840 против 0,795 у скоринговой карты, но разброс по окнам около 0,09, так что выигрыш небольшой. На validation модели почти равны (Gini 0,969 у обеих; 13 банкротов). Калибровка isotonic снижает Brier с 0,0114 до 0,0083, но на validation это оценка «в выборке». Новые признаки на validation Gini не подняли (0,974 без них, 0,969 с ними), разница в пределах шума.

# Часть 4. 04. Финансовая оценка, интерпретация, стабильность (этап 4)
Test (out-of-time) открывается **один раз** — внутри `P.step4_evaluation()`. Здесь же: Gini и полнота при фиксированной доле проверок, прибыль портфеля (LGD = 45%), SHAP и разбор трёх наблюдений, PSI, сохранение `model.pkl`.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
root = os.getcwd()                              # ищем корень проекта (где лежит config.yaml) — работает и в Colab
while not os.path.exists(os.path.join(root, "config.yaml")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, root); os.chdir(root)
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.float_format", "{:.4f}".format)
from src import pipeline as P
c = P.context()

In [ ]:
r4 = P.ensure("step4")
print("движок:", r4["engine"], "| метод объяснений:", r4["shap_method"], "| test:", r4["test_n"], "компаний,", r4["test_bankrupt"], "банкротств")

## 1. Метрики варианта на out-of-time test

In [ ]:
def row(d, name):
    return {"модель": name, "Gini": d["gini"], "Gini ДИ95": tuple(round(x, 3) for x in d["gini_ci"]), "KS": d["ks"], "PR-AUC": d["pr_auc"],
            **{f"полнота@{s}%": d[f"recall@{s}%"] for s in (5, 10, 20)}, "Brier (калибр.)": d["brier_calibrated"]}
pd.DataFrame([row(r4["base"], "скоринговая карта"), row(r4["boost"], "бустинг")])

In [ ]:
print("Gini(бустинг) − Gini(карта):", r4["gini_diff"])

In [ ]:
P.show_fig("fig_calibration_test.png")

## 2. Перевод качества в деньги
Одобряем компанию, если PD < порога. Доход с хорошей одобренной компании — `margin × EAD` (маржа 8% — **допущение**, в данных нет сумм), потери с одобренной обанкротившейся — `LGD × EAD`, LGD = 45%. Порог выбран на validation по максимуму **ожидаемой** прибыли по калиброванным PD; на test считаем реализованную прибыль.

In [ ]:
f = r4["finance"]
tab = pd.DataFrame({k: {x: f[k][x] for x in ("approve_rate", "good_approved", "bad_approved", "income", "loss", "profit", "profit_per_exposure")}
                    for k in ("all_approved", "boosting", "baseline", "oracle")}).T
print("порог PD (бустинг):", round(f["threshold_pd"], 4), "| порог PD (карта):", round(f["threshold_pd_baseline"], 4))
tab

In [ ]:
pd.DataFrame(f["sensitivity"])

In [ ]:
P.show_fig("fig_finance.png")

## 3. SHAP: summary plot и разбор трёх наблюдений

In [ ]:
P.show_fig("fig_shap_summary.png")

In [ ]:
for cs in r4["cases"]:
    print(f'--- {cs["title"]} | PD={cs["pd_calibrated"]:.1%} | факт: {"банкрот" if cs["target"] else "не банкрот"} | решение: {cs["decision"]}')
    print(pd.DataFrame(cs["top"]).to_string(index=False))
P.show_fig("fig_shap_cases.png")

## 4. Стабильность: PSI train → validation / test
PSI > 0,25 — признак нестабилен; 0,10–0,25 — умеренный сдвиг.

In [ ]:
pd.DataFrame(r4["psi"])

In [ ]:
P.show_fig("fig_psi.png")

## 5. Сохранённая модель и `predict(df)`

In [ ]:
import pandas as pd
from src.predict import predict
raw = pd.read_csv("data/data.csv")
print(predict(raw.tail(5), return_decision=True))
print("predict(df) совпадает с оценкой в пайплайне:", r4["predict_check_ok"])

**Вывод.** На test (1023 компании, 19 банкротов) Gini бустинга 0,935, карты 0,931; разница 0,004, доверительный интервал [−0,016; 0,029] — модели статистически неразличимы. Полнота при проверке 10% компаний: 79% у бустинга и 84% у карты. Стратегия с порогом PD даёт у бустинга 7,42% прибыли от экспозиции против 7,02% при «одобрить всех» (маржа 8% — допущение). По SHAP главные драйверы — retained_self_financing и показатели рентабельности; PSI скора 0,031 (стабилен), из топ-5 признаков нестабилен только Quick Ratio (PSI 0,33).

## 5. Отчёт и скачивание результатов
Отчёт `reports/report.docx` (5–7 страниц) собирается кодом из **реальных** результатов запуска, поэтому все числа в нём совпадают с таблицами выше. Откройте его и дополните выводы своими словами.

In [ ]:
from src.report import build_report
rep = build_report()
print("отчёт:", rep)
import json
r = json.load(open("artifacts/results.json", encoding="utf-8"))
print("движок:", r["step4"]["engine"], "| метод объяснений:", r["step4"]["shap_method"])
assert r["step4"]["engine"] == "lightgbm" and r["step4"]["shap_method"] == "shap", "Отчёт предварительный: нужны lightgbm и shap (см. ячейку 0.1)"

In [ ]:
# Скачиваем: отчёт, модель, архив проекта (для Git-репозитория)
import shutil
from google.colab import files
shutil.make_archive("/content/bankruptcy-ml-result", "zip", "/content/bankruptcy-ml")
files.download("reports/report.docx")
files.download("model.pkl")
files.download("/content/bankruptcy-ml-result.zip")